# Barbershop-FT: ฝึก shared weights 4 trials × 30 epochs แล้วเทียบ HairFastGAN
**ฉบับแก้: มีการฝึกน้ำหนักเครือข่ายด้วย train150 จริง ไม่ใช่เพียงค้นหา LR ของ latent optimizer**

เปิด Colab GPU แล้วรันตามลำดับทั้งไฟล์ เตรียม review ZIP ของ A–D, `HairFastGAN_Dataset_Split_v2.zip` และ Drive checkpoint/cache/inputs ของ HairFastGAN เดิม เปลี่ยน `STUDY_ROOT` ถ้าย้ายโฟลเดอร์

ส่วนแรกประเมิน HairFastGAN baseline และ A–D เดิม ไม่ฝึกซ้ำ ส่วนหลังฝึก **Barbershop-FT** ใหม่ 4 trials จาก pretrained เดียวกัน (epoch0 ไม่ใช่ epoch27 เดิม) ใช้ 150 คู่เดิม × 30 epochs, batch1 = **4500 shared-weight updates ต่อ trial** และ validation9 คู่เดิม จากนั้นล็อก best checkpoints ทั้งสี่และทดสอบ baseline+A–D บน test10 คู่เดียวกัน

## วิธีที่ดัดแปลง — ต้องใช้ชื่อนี้ในรายงานวิจัย
Barbershop เดิมเป็นวิธี GAN inversion/latent optimization ไม่มี training loop แบบ HairFastGAN จึงดัดแปลงเป็น **frozen latent front end + shared synthesis fine-tuning**:
1. ใช้ pretrained เดิมทำ inversion/alignment/blending เพื่อเตรียม latent ของแต่ละคู่ครั้งเดียว โดยตรึง noise เพื่อให้ทำซ้ำได้
2. ฝึกน้ำหนัก convolution และ ToRGB ของ generator ส่วน 64–1024px ด้วย train150 เท่านั้น latent ไม่ได้ถูกฝึกใน loop นี้
3. ใช้ validation เลือก best epoch/ตัวเลือก ห้ามเลือกจาก test การ optimize latent ของ test เป็นขั้นประมวลผลรายภาพของ Barbershop ไม่อัปเดต shared weights

**นี่เป็นวิธีวิจัยดัดแปลงที่ยังไม่ยืนยันคุณภาพ ไม่ใช่ขั้นตอน fine-tune ที่ผู้พัฒนา Barbershop รับรอง** ดูต้นฉบับ: https://github.com/ZPdesu/Barbershop

| เงื่อนไข | สิ่งที่ใช้ร่วมกัน |
|---|---|
| ข้อมูล | train150 / validation9 / test10 และคู่ภาพเดิม |
| งบฝึก shared weights | 30 epochs, batch1, 4500 updates/trial |
| Trials | 4 ชุด; LR ตัวเลขตาม A–D เดิม อ่านจาก manifest |
| Optimizer | Adam, seed เดิม, clip norm1 |
| การประเมิน | คู่ภาพ/masks/ฟังก์ชันวัดร่วมกัน บน runtime เดียวกัน |

**สิ่งที่ยังต่างตามสถาปัตยกรรม:** กลุ่มพารามิเตอร์, จำนวนพารามิเตอร์, compute, pretrained data และ loss ทั้งชุด กลุ่ม LR ของ BS-FT คือ synthesis/ToRGB ไม่ใช่ Blending/PostProcess ของ HFG ไม่มีการอ้างว่าทุกเงื่อนไขหรือ FLOPs เท่ากัน

Loss ของ BS-FT ใช้ final color/shape, identity, protected และ detail โดยค่าน้ำหนักอ่านจากงานเดิม ตัด pre-stage/latent-anchor ของ HFG ที่ไม่ตรงกับวิธีนี้ออก และรักษา detail เทียบ baseline ของ BS เอง Gate detail จึงเป็น absolute MAE ≤ baseline+0.02 (ตั้งก่อนรัน) กราฟข้ามระบบไม่เปรียบเทียบค่า detail ที่ใช้คนละภาพอ้างอิง

**ข้อจำกัดสำคัญ:** shape target ยังมาจาก HFG จึงเป็น target agreement ไม่ใช่ GT อิสระ; test เป็น candidate holdout ยังไม่ยืนยันการซ้ำกับทุกชุดพัฒนาเก่า; SD ไม่ใช่ confidence interval ไม่ควรอ้างความเหนือกว่าทั่วไปจากชุดเล็กนี้

เตรียม latent 159 คู่ก่อนฝึก และ 10 คู่หลังล็อก checkpoint อาจใช้เวลานานมาก งานเตรียมบันทึกเป็นรายคู่ งานฝึก resume จาก **จบ epoch ล่าสุด** ถ้าหลุดกลาง epoch จะทำ epoch นั้นใหม่ CSV/optimizer กู้จาก last.pt เพื่อไม่ให้นับ updates ซ้ำ

ผลลัพธ์: best.pt / last.pt / final_epoch30.pt ต่อ trial, gradient/weight-change proof, train/val CSV, กราฟแยกและรวม, baseline+A–D test, ภาพเทียบ HFG+BS และ PDF/ZIP รายงาน Checkpoints และ cache อยู่ใน Drive

**การตรวจ:** ตรวจ syntax และโครงสร้างโค้ดในเครื่องผู้ช่วย ยังไม่ได้รัน GPU end-to-end; notebook มี preflight ตรวจผล forward/gradient/น้ำหนักเปลี่ยนจริงก่อนเริ่ม 30 epochs ถ้าไม่ผ่านจะหยุด ไม่ข้าม assertion หรือเพิ่ม tolerance อัตโนมัติ


In [ ]:
%pip -q install numpy pandas matplotlib


In [ ]:
from google.colab import files
from pathlib import Path
import json, zipfile, uuid, pandas as pd
uploaded=files.upload()  # เลือก review ZIP หนึ่งไฟล์เท่านั้น
if len(uploaded)!=1:raise ValueError('เลือก review ZIP เพียงหนึ่งไฟล์')
REVIEW_ZIP=Path(next(iter(uploaded))).resolve()
if not zipfile.is_zipfile(REVIEW_ZIP):raise ValueError('ไม่ใช่ ZIP ที่อ่านได้')
LOCAL_REPORT=Path('/content')/('ABCD_complete_report_'+uuid.uuid4().hex[:10])
LOCAL_REPORT.mkdir(exist_ok=False)
def zip_member(z,name):
    matches=[n for n in z.namelist() if n==name or n.endswith('/'+name)]
    if len(matches)!=1:raise ValueError('ZIP ต้องมีไฟล์เดียวสำหรับ '+name)
    return matches[0]
with zipfile.ZipFile(REVIEW_ZIP) as z:
    ZIP_MANIFESTS={t:json.loads(z.read(zip_member(z,f'trial_{t}/manifest.json'))) for t in 'ABCD'}
    ZIP_BOARD=pd.read_csv(z.open(zip_member(z,'leaderboard.csv'))).set_index('trial')
    for t,m in ZIP_MANIFESTS.items():
        tr=pd.read_csv(z.open(zip_member(z,f'trial_{t}/training_steps.csv')))
        v=pd.read_csv(z.open(zip_member(z,f'trial_{t}/metrics.csv')))
        v=v[(v.epoch>=0)&(v.split=='validation')]
        tc=tr.groupby('epoch').size();vc=v.groupby('epoch')['case'].nunique()
        assert tc.index.tolist()==list(range(1,31)) and (tc==150).all() and tr['update'].nunique()==4500, f'{t}: training log incomplete'
        assert vc.index.tolist()==list(range(31)) and (vc==9).all() and not v.duplicated(['epoch','case']).any(), f'{t}: validation log incomplete'
        assert int(ZIP_BOARD.loc[t,'best_epoch'])==int(m['best_epoch'])
        print(t,': ZIP logs complete, best epoch',m['best_epoch'])


In [ ]:
"""Plot recorded HairFastGAN trials; no training or checkpoint loading.
CLI: python plot_train_val.py --source results.zip --out figures
Also accepts an extracted study directory containing trial_A ... trial_D.
"""
from pathlib import Path
import argparse
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

COLORS = dict(A='#3178C6', B='#098C79', C='#DB9031', D='#9966B3')
SPECS = [
 ('loss', 'Total loss ↓'),
 ('L_quantile', 'Lightness error L ↓'),
 ('ab_quantile', 'Color error ab ↓'),
 ('identity_cosine', 'Face identity cosine ↑'),
 ('protected_RGB_MAE_vs_input', 'Protected-region MAE ↓'),
 ('detail_512_MAE_vs_pre', 'Detail MAE vs pre-output ↓'),
 ('color_loss', 'Color loss ↓'),
 ('shape_loss', 'Shape loss ↓'),
 ('target_mask_IoU', 'Hair-mask IoU ↑'),
]
COMPARE = [SPECS[1], SPECS[2], SPECS[8], SPECS[3], SPECS[4], ('score', 'Combined color / shape score ↓')]

class Source:
    def __init__(self, path):
        self.path = Path(path)
        self.z = zipfile.ZipFile(self.path) if self.path.is_file() else None
    def read_csv(self, name):
        if self.z:
            matches = [n for n in self.z.namelist() if n == name or n.endswith('/' + name)]
            if len(matches) != 1:
                raise ValueError(f'Expected one {name}; found {matches}')
            with self.z.open(matches[0]) as f:
                return pd.read_csv(f)
        return pd.read_csv(self.path / name)
    def close(self):
        if self.z: self.z.close()

def style(ax, title):
    ax.set_title(title, loc='left', fontsize=12, fontweight='bold', pad=10)
    ax.grid(alpha=.2)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Epoch')
    ax.set_xlim(0, 30)
    ax.set_xticks(range(0, 31, 5))

def run(source, out='figures', show=True):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    src = Source(source)
    plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':10, 'figure.facecolor':'white'})
    histories = {}; trains = {}; selections = {}; counts = {}
    try:
        leaderboard = src.read_csv('leaderboard.csv').set_index('trial')
        for t in COLORS:
            steps = src.read_csv(f'trial_{t}/training_steps.csv')
            records = src.read_csv(f'trial_{t}/metrics.csv')
            # Exclude the old comparator (negative epochs), never use its results.
            records = records.loc[(records['split'] == 'validation') & (records.epoch >= 0)].copy()
            steps = steps.loc[steps.epoch >= 1].copy()
            if records.duplicated(['epoch', 'case']).any() or steps['update'].duplicated().any():
                raise ValueError(f'{t}: duplicated records; resolve resumed-run logs first')
            val = records.groupby('epoch').mean(numeric_only=True)
            train = steps.groupby('epoch').mean(numeric_only=True)
            if val.index.tolist() != list(range(31)) or train.index.tolist() != list(range(1,31)):
                raise ValueError(f'{t}: expected validation 0..30 and training 1..30')
            sizes = records.groupby('epoch').size()
            if sizes.nunique() != 1:
                raise ValueError(f'{t}: validation count changes between epochs')
            counts[t] = int(sizes.iloc[0])
            selections[t] = int(leaderboard.loc[t, 'best_epoch'])
            histories[t], trains[t] = val, train
            train.to_csv(out / f'trial_{t}_train_epoch_means.csv')
            val.to_csv(out / f'trial_{t}_validation_epoch_means.csv')
    finally:
        src.close()
    base = histories['A'].loc[0]
    for t,val in histories.items():
        metric_keys=[k for k,_ in SPECS]
        np.testing.assert_allclose(val.loc[0,metric_keys],base[metric_keys],rtol=1e-7,atol=1e-9)
        # Same normalized score used in the supplied leaderboard. Baseline = 1.
        val['score'] = .5*((val.L_quantile+val.ab_quantile)/(base.L_quantile+base.ab_quantile)
                          +(1-val.target_mask_IoU)/(1-base.target_mask_IoU))
        np.testing.assert_allclose(val.loc[selections[t],'score'],leaderboard.loc[t,'score'],rtol=1e-7)
    figures = []
    def save(fig, name, pdf):
        fig.savefig(out / (name+'.png'),dpi=180,bbox_inches='tight')
        pdf.savefig(fig,bbox_inches='tight')
        figures.append(out / (name+'.png'))
        if show: plt.show()
        plt.close(fig)
    with PdfPages(out/'all_train_val_and_comparisons.pdf') as pdf:
        # Display A, then B, C, D before any combined figure.
        for t in COLORS:
            train, val = trains[t], histories[t]; best = selections[t]
            fig, axs = plt.subplots(3,3,figsize=(15,12))
            for ax,(key,title) in zip(axs.flat,SPECS):
                if key in train:
                    ax.plot(train.index,train[key],color='#3178C6',lw=1.8,label='Train: mean of update logs')
                else:
                    ax.text(.04,.92,'Train metric not logged',transform=ax.transAxes,va='top',fontsize=9,color='#667085')
                ax.plot(val.index,val[key],color='#D87523',lw=1.8,label='Validation: end of epoch')
                ax.axhline(base[key],ls=':',color='#7B8494',lw=1.2,label='Pretrained validation baseline')
                ax.axvline(best,ls='--',color='#818898',lw=1,alpha=.6)
                ax.scatter(best,val.loc[best,key],marker='*',s=130,color='#D87523',edgecolor='white',zorder=5)
                style(ax,title)
            fig.suptitle(f'Trial {t} | Train and validation | Selected epoch {best}',fontsize=20,fontweight='bold',y=.985)
            fig.text(.07,.948,f'Blending LR = {leaderboard.loc[t,"blend_lr"]:g}    PostProcess LR = {leaderboard.loc[t,"pp_lr"]:g}    Validation: {counts[t]} pairs',fontsize=11)
            h,l=axs.flat[0].get_legend_handles_labels()
            fig.legend(h,l,loc='lower center',bbox_to_anchor=(.5,.052),ncol=3,frameon=False)
            fig.text(.07,.02,'Train values are recorded during weight updates; validation uses the end-of-epoch model.\nTheir gap is descriptive, not a controlled train-set vs validation-set evaluation. Star = selected checkpoint.',fontsize=10,color='#667085')
            fig.subplots_adjust(top=.89,bottom=.13,hspace=.4,wspace=.29)
            save(fig,f'01_trial_{t}_train_validation',pdf)
        # Combined train/validation: separate columns; common y-scale per metric.
        fig,axs=plt.subplots(3,2,figsize=(13,12),sharex=True)
        for i,(key,title) in enumerate([SPECS[0],SPECS[1],SPECS[2]]):
            limits=[]
            for j,(collection,label) in enumerate([(trains,'Train: update-log means'),(histories,'Validation: end of epoch')]):
                ax=axs[i,j]
                for t,d in collection.items(): ax.plot(d.index,d[key],color=COLORS[t],label=t,lw=1.7)
                style(ax,f'{title} | {label}')
                limits.extend(ax.get_ylim())
            for ax in axs[i]: ax.set_ylim(min(limits),max(limits))
        fig.suptitle('All trials | Train and validation in separate panels',fontsize=19,fontweight='bold')
        fig.legend(*axs[0,0].get_legend_handles_labels(),loc='lower center',ncol=4,frameon=False)
        fig.tight_layout(rect=(0,.04,1,.95));save(fig,'02_all_trials_train_validation',pdf)
        # Same validation comparison layout as the earlier report.
        fig,axs=plt.subplots(2,3,figsize=(15,9))
        for ax,(key,title) in zip(axs.flat,COMPARE):
            ax.axhline(histories['A'].loc[0,key],ls='--',color='#7B8494',label='Baseline')
            for t,val in histories.items():
                ax.plot(val.index,val[key],color=COLORS[t],label=t,lw=1.7)
                e=selections[t];ax.scatter(e,val.loc[e,key],marker='*',s=120,color=COLORS[t],edgecolor='white',zorder=5)
            style(ax,title)
        fig.suptitle('All trials | Validation learning curves',fontsize=20,fontweight='bold')
        fig.legend(*axs.flat[0].get_legend_handles_labels(),loc='lower center',ncol=5,frameon=False)
        fig.tight_layout(rect=(0,.06,1,.95));save(fig,'03_all_trials_validation',pdf)
        rows=[dict(model='Baseline',epoch=0,**histories['A'].loc[0].to_dict())]
        for t,val in histories.items():rows.append(dict(model=t,epoch=selections[t],**val.loc[selections[t]].to_dict()))
        result=pd.DataFrame(rows)
        result.to_csv(out/'best_checkpoints_vs_baseline.csv',index=False)
        fig,axs=plt.subplots(2,3,figsize=(15,9))
        for ax,(key,title) in zip(axs.flat,COMPARE):
            values=result[key].to_numpy()
            ax.bar(range(5),values,color=['#7B8494']+list(COLORS.values()),width=.65)
            ax.set_xticks(range(5),['Baseline']+[f'{t}\nep {selections[t]}' for t in COLORS])
            ax.set_ylim(0,max(values)*1.2)
            ax.set_title(title,loc='left',fontsize=12,fontweight='bold');ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True)
            ax.spines[['top','right']].set_visible(False)
            for i,v in enumerate(values):ax.text(i,v+max(values)*.025,f'{v:.4f}',ha='center',fontsize=10)
        fig.suptitle('Official pretrained baseline vs best checkpoint of A–D',fontsize=19,fontweight='bold')
        fig.text(.06,.025,f'Validation means ({counts["A"]} pairs); not test results. Baseline = epoch 0. Negative-epoch comparator rows excluded.',color='#667085')
        fig.tight_layout(rect=(0,.07,1,.95));save(fig,'04_baseline_vs_best_trials',pdf)
    notes = {
        'source':str(source), 'selected_epochs':selections, 'validation_pairs':counts,
        'train_aggregation':'Unweighted arithmetic mean of update logs per epoch (batch=1 in this study).',
        'validation_aggregation':'Arithmetic mean across validation cases at each epoch.',
        'missing_train_metrics':['target_mask_IoU'],
        'score_formula':'0.5*((L+ab)/(baseline_L+baseline_ab)+(1-IoU)/(1-baseline_IoU))',
        'selection':'Uses existing leaderboard best_epoch, not separate best epoch for each metric.',
        'excluded':'All validation rows with epoch < 0 and all old comparator columns.',
        'caution':'Single-seed validation results; no test evaluation or confidence intervals inferred.'
    }
    (out/'plot_notes.json').write_text(json.dumps(notes,indent=2),encoding='utf-8')
    print('Saved figures and tables to:',out.resolve())
    return result, figures



In [ ]:
history_comparison,history_figures=run(REVIEW_ZIP,LOCAL_REPORT/'01_train_val',show=True)
print('Train/validation plots ready:',LOCAL_REPORT/'01_train_val')


## ประเมินโมเดลใหม่ด้วยไฟล์ใน Drive
ส่วนนี้ต้องใช้ GPU และไฟล์ checkpoint/cache เดิม ไม่ได้อ่านสถานะ running จาก Drive มาใช้

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('กรุณาเลือก Runtime > Change runtime type > T4 GPU แล้ว Run all อีกครั้ง')
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download

drive.mount('/content/drive')
LOCAL_REPO = '/content/HairFastGAN'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
RESULTS_ROOT = Path('/content/drive/MyDrive/HairFastGAN_prompt_study')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
        shutil.copytree(local_models, DRIVE_CACHE_DIR, dirs_exist_ok=True)
# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))

# Small on-device smoke check before loading model weights.
import torch
_check = {'torch':torch, 'F':torch.nn.functional}
exec(FUSED_NATIVE + '\n' + UP_NATIVE_IMPL + '\n' + UP_NATIVE, _check)
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_x = torch.randn(1, 2, 5, 5, device=_device, requires_grad=True)
_b = torch.randn(2, device=_device, requires_grad=True)
_y = _check['fused_leaky_relu'](_x, _b, .1)
torch.testing.assert_close(_y, torch.nn.functional.leaky_relu(_x + _b.view(1,2,1,1), .1) * (2 ** .5))
_z = _check['upfirdn2d'](_x, torch.ones(1,1,device=_device))
torch.testing.assert_close(_z, _x)
(_y.sum() + _z.sum()).backward()
assert torch.isfinite(_x.grad).all() and torch.isfinite(_b.grad).all()
print('Native operator smoke check passed on', _device)
del _check, _x, _b, _y, _z


In [ ]:
from PIL import Image
if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')


In [ ]:
import io, csv, zipfile, hashlib, uuid, random, copy, traceback, time, gc
import numpy as np
import pandas as pd
import torch.nn.functional as F
from PIL import Image, ImageDraw
from IPython.display import display
from models.Net import get_segmentation as segment
from torch.utils.checkpoint import checkpoint

# แก้เพียง path นี้หากย้ายโฟลเดอร์ผลการทดลอง
STUDY_ROOT = Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart/9be83b9c2e')
OUT = LOCAL_REPORT/'02_reevaluation'
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()
MANIFESTS=ZIP_MANIFESTS  # ZIP is the authoritative history; do not read Drive status
REFERENCE_META=MANIFESTS['A']
SOURCE=Path(REFERENCE_META['source'])
VAL_KEYS=REFERENCE_META['config']['validation_keys']
assert len(VAL_KEYS)==9 and len(set(VAL_KEYS))==9
bykey={k:{'split':'validation'} for k in VAL_KEYS}
for t,m in MANIFESTS.items():
    assert m['config']['validation_keys']==VAL_KEYS
    assert m['repo_commit']==REPO_COMMIT
    assert (STUDY_ROOT/f'trial_{t}/best.pt').is_file()
    for k in VAL_KEYS:
        assert sha(SOURCE/'cache'/f'{k}.pt')==m['source_cache_hashes'][k], f'Cache changed: {t}/{k}'
        for mask in ['eval_core','eval_refmask','target','protected']:
            a=STUDY_ROOT/'trial_A/inputs'/k/(mask+'.png')
            other=STUDY_ROOT/f'trial_{t}/inputs'/k/(mask+'.png')
            assert sha(a)==sha(other), f'Masks differ: {t}/{k}/{mask}'
b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
assert sha(b.opts.blending_checkpoint)==REFERENCE_META['init_blending_sha256']
assert sha(b.opts.pp_checkpoint)==REFERENCE_META['init_postprocess_sha256']
for module in [enc,G,pp,align]: module.eval().requires_grad_(False)
WEIGHTS=REFERENCE_META['config']['weights']
OUT.mkdir(parents=True,exist_ok=False)
print('Validation cases:',len(VAL_KEYS),'| New output:',OUT)

# Verify every checkpoint before expensive evaluation; never modify source files.
for t in 'ABCD':
    checkpoint_path=STUDY_ROOT/f'trial_{t}/best.pt'
    ck=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    assert ck['format']=='hairfast_bpp_pretrained_start_trial_v1', f'{t}: format mismatch'
    assert ck['config']['trial']==t, f'{t}: wrong trial checkpoint'
    assert int(ck['epoch'])==int(MANIFESTS[t]['best_epoch']), f'{t}: checkpoint epoch differs from ZIP; select the matching study directory'
    for field in ['blend_lr','pp_lr','seed','weights','train_keys','validation_keys']:
        assert ck['config'][field]==MANIFESTS[t]['config'][field], f'{t}: config mismatch: {field}'
    assert ck['source_manifest_sha256']==sha(SOURCE/'manifest.json')
    with zipfile.ZipFile(REVIEW_ZIP) as z:
        zip_study=json.loads(z.read(zip_member(z,'study.json')))
    assert ck['mask_zip_sha256']==zip_study['mask_zip_sha256']
    del ck
print('Checkpoint checks passed; no training will run')


In [ ]:
runtime_baseS={}
def cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple((cpu(v) for v in x))
    return copy.deepcopy(x)

def gpu(x):
    if torch.is_tensor(x):
        return x.to('cuda').clone()
    if isinstance(x, dict):
        return {k: gpu(v) for k, v in x.items()}
    if isinstance(x, list):return [gpu(v) for v in x]
    if isinstance(x, tuple):return tuple(gpu(v) for v in x)
    return x

def resize(x):
    return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

def saveim(x, p):
    a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
    Image.fromarray((a * 255).round().astype('uint8')).save(p)

def lab(x):
    x = x.float().clamp(0, 1)
    lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
    M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
    xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
    return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)

def render(s, d):
    S = torch.cat([d['S'][:, :6], s], 1)
    return G([S], input_is_latent=True, return_latents=False, start_layer=4, end_layer=8, layer_in=d['F'], randomize_noise=False)[0]

def finish(raw, d):
    sf, ff = pp(d['I1'], b.downsample_256(raw))
    return G([sf], input_is_latent=True, return_latents=False, start_layer=5, end_layer=8, layer_in=ff, randomize_noise=False)[0]

def detail(x):
    y = x.mean(1, keepdim=True)
    mu = F.avg_pool2d(y, 9, 1, 4)
    v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
    return (y - mu) / (v + 0.001).sqrt()

def unit(raw):return resize(((raw+1)/2).clamp(0,1))
def read_mask(p):
    with Image.open(p) as im:
        if im.size!=(256,256):raise ValueError('Mask size mismatch')
        a=np.asarray(im.convert('L'))>127
    return torch.from_numpy(a.copy())[None,None]

def load_case(k):
    d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
    folder=STUDY_ROOT/'trial_A/inputs'/k
    for name in ['eval_core','eval_refmask','target','protected']:
        mask=read_mask(folder/(name+'.png'))
        if name in ['target','protected']:
            assert torch.equal(d[name].bool(),mask), (k,name)
        d[name]=mask
    for name in ['core','refmask']:d[name]=d[name].bool()
    for name in ['eval_core','eval_refmask']:assert d[name].sum()>=100
    return d

def quantile_color(im,ref,mask,refmask):
    l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
    q=torch.linspace(.1,.9,9,device=im.device)
    diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
    return diff[:,0].mean(),diff[:,1:].mean()

def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))

def shape_metrics(im,d):
    labels=segment(align.to_bisenet(im))
    pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
    target=d['target']
    return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))


# Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
from losses.pp_losses import Backbone
from huggingface_hub import hf_hub_download
id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
if not id_path.is_file():
    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
id_net=id_net.cuda().eval().requires_grad_(False)
FaceParsing_tensor.parsing_img()
parser_net=FaceParsing.bise_net
if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
parser_net.eval().requires_grad_(False)
assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13

def id_features(im):
    x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
    feats=id_net(x)
    return F.normalize(feats,dim=1)

def constraint_terms(im,d,fx):
    # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
    logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
    other=torch.cat((logits[:,:17],logits[:,18:]),1)
    binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
    target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
    if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
    prob=binary.sigmoid()
    bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
    balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
    dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
    embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
    identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
    return identity,balanced+dice,balanced,dice

def to_size(x,size):
    return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)

def detail_terms(full,d,fx):
    values={}
    for size in [256,512]:
        x=to_size(full,size)
        teacher=fx['pre'] if size==256 else fx['pre_512']
        mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
        inner=-F.max_pool2d(-mask,9,1,4)
        if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
        values[size]=masked_mae(detail(x),detail(teacher),inner)
    return values


from scipy.ndimage import binary_erosion,binary_dilation
from skimage.metrics import structural_similarity
_basic_shape_metrics=shape_metrics
def shape_metrics(im,d):
    result=_basic_shape_metrics(im,d)
    labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
    t=d['target'][0,0].cpu().numpy()>.5
    result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
    a=p^binary_erosion(p);bnd=t^binary_erosion(t)
    prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
    rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
    result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
    return result

def extra_metrics(im,d):
    x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
    mask=d['protected'][0,0].cpu().numpy()>.5
    _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
    safe=binary_erosion(mask,iterations=3)
    if not safe.any():raise ValueError('No protected SSIM interior')
    mse=float(((x-y)**2)[mask].mean())
    return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
        any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


In [ ]:
def joint_forward(d,grad=False):
    def blend_fn(a,c,x,y):return enc(a,c,x,y)
    args=[d['args'][i] for i in range(4)]
    s=checkpoint(blend_fn,*args,use_reentrant=False) if grad else blend_fn(*args)
    raw_pre=checkpoint(lambda a:render(a,d),s,use_reentrant=False) if grad else render(s,d)
    pp_in=b.downsample_256(raw_pre)
    sf,ff=checkpoint(lambda a,c:pp(a,c),d['I1'],pp_in,use_reentrant=False) if grad else pp(d['I1'],pp_in)
    def generate(a,c):return G([a],input_is_latent=True,return_latents=False,
        start_layer=5,end_layer=8,layer_in=c,randomize_noise=False)[0]
    raw=checkpoint(generate,sf,ff,use_reentrant=False) if grad else generate(sf,ff)
    return raw_pre,raw,s,sf,ff

def objective(raw_pre,raw,s,sf,ff,d,fx):
    full=((raw+1)/2).clamp(0,1);im=to_size(full,256);pre=unit(raw_pre)
    L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
    preL,preab=quantile_color(pre,d['ref'],d['eval_core'],d['eval_refmask'])
    identity,shape,_,_=constraint_terms(im,d,fx)
    _,preshape,_,_=constraint_terms(pre,d,fx)
    protected=masked_mae(im,d['face'],d['protected'])
    details=detail_terms(full,d,fx);texture=torch.stack(list(details.values())).mean()
    anchor=(s-fx['s']).square().mean()+(sf-fx['sf']).square().mean()+(ff-fx['ff']).square().mean()
    color=(L+ab)/100;precolor=(preL+preab)/100
    loss=(WEIGHTS['final_color']*color+WEIGHTS['final_shape']*shape+
          WEIGHTS['pre_color']*precolor+WEIGHTS['pre_shape']*preshape+
          WEIGHTS['identity']*identity+WEIGHTS['protected']*protected+
          WEIGHTS['detail']*texture+WEIGHTS['anchor']*anchor)
    return loss,dict(loss=loss,color_loss=color,shape_loss=shape,pre_color_loss=precolor,
        pre_shape_loss=preshape,identity_loss=identity,identity_cosine=1-identity,
        protected_RGB_MAE_vs_input=protected,detail_512_MAE_vs_pre=details[512],
        detail_MAE_vs_pre=details[256],detail_training_term=texture,latent_anchor=anchor,
        L_quantile=L,ab_quantile=ab,pre_L_quantile=preL,pre_ab_quantile=preab)

In [ ]:
def reset_pretrained():
    sd=torch.load(b.opts.blending_checkpoint,map_location='cpu',weights_only=True)['model_state_dict']
    missing,unexpected=enc.load_state_dict(sd,strict=False)
    if unexpected or any(not k.startswith('clip_model.') for k in missing):
        raise RuntimeError('Pretrained Blending schema mismatch')
    pp.load_state_dict(torch.load(b.opts.pp_checkpoint,map_location='cpu',weights_only=True)['model_state_dict'],strict=True)
    enc.eval();pp.eval()

# Compute shared baseline detail/latent references once from original pretrained weights.
random.seed(3407);np.random.seed(3407);torch.manual_seed(3407);torch.cuda.manual_seed_all(3407)
reset_pretrained()
fixed={}
with torch.no_grad():
    for k in VAL_KEYS:
        d=gpu(load_case(k));pre,raw,s,sf,ff=joint_forward(d)
        fixed[k]=cpu(dict(pre=unit(pre),pre_512=to_size(((pre+1)/2).clamp(0,1),512),s=s,sf=sf,ff=ff,face_embedding=id_features(d['face'])))
        del d,pre,raw,s,sf,ff

rows=[];provenance={};epochs={'Baseline':0}
with torch.no_grad():
    for model in ['Baseline','A','B','C','D']:
        reset_pretrained() # Every model starts with an explicit complete reset.
        if model!='Baseline':
            path=STUDY_ROOT/f'trial_{model}/best.pt'
            ck=torch.load(path,map_location='cpu',weights_only=True)
            assert ck['format']=='hairfast_bpp_pretrained_start_trial_v1'
            assert ck['config']['trial']==model
            assert ck['repo_commit']==REPO_COMMIT
            assert ck['config']['validation_keys']==VAL_KEYS
            assert ck['source_manifest_sha256']==sha(SOURCE/'manifest.json')
            assert ck['config']['weights']==WEIGHTS
            assert int(ck['epoch'])==int(MANIFESTS[model]['best_epoch'])
            assert ck['start_blending_sha256']==REFERENCE_META['init_blending_sha256']
            sd=ck['blending_trainable_state_dict']
            expected={n for n,p in enc.named_parameters() if n.startswith('modulation_module_list.')}
            assert set(sd)==expected
            state=enc.state_dict();state.update(sd);enc.load_state_dict(state,strict=True)
            pp.load_state_dict(ck['post_process_state_dict'],strict=True)
            epochs[model]=int(ck['epoch'])
            provenance[model]={'checkpoint':str(path),'sha256':sha(path),'epoch':epochs[model]}
            del ck,state,sd
        else:
            provenance[model]={'blending_sha256':sha(b.opts.blending_checkpoint),'postprocess_sha256':sha(b.opts.pp_checkpoint),'epoch':0}
        for k in VAL_KEYS:
            d=gpu(load_case(k));fx=gpu(fixed[k])
            pre,raw,s,sf,ff=joint_forward(d)
            _,metrics=objective(pre,raw,s,sf,ff,d,fx)
            im=unit(raw)
            metrics.update(shape_metrics(im,d));metrics.update(extra_metrics(im,d))
            vals={key:float(value) for key,value in metrics.items()}
            if not all(np.isfinite(v) for v in vals.values()):raise ValueError((model,k,'Non-finite metrics'))
            rows.append(dict(model=model,epoch=epochs[model],case=k,split='validation',**vals))
            folder=OUT/'images'/k;folder.mkdir(parents=True,exist_ok=True)
            saveim((raw+1)/2,folder/(model+'.png'))
            if model=='Baseline':
                for field in ['face','aligned','ref']:saveim(d[field],folder/(field+'.png'))
            del d,fx,pre,raw,s,sf,ff,metrics,im
            print(model,k,'done',flush=True)
        pd.DataFrame(rows).to_csv(OUT/'per_case_metrics.csv',index=False)
reset_pretrained()
frame=pd.DataFrame(rows)
summary=frame.groupby('model').mean(numeric_only=True).reindex(['Baseline','A','B','C','D'])
base=summary.loc['Baseline']
summary['score']=.5*((summary.L_quantile+summary.ab_quantile)/max(base.L_quantile+base.ab_quantile,1e-8)+(1-summary.target_mask_IoU)/max(1-base.target_mask_IoU,1e-8))
summary.to_csv(OUT/'summary_metrics.csv')
(OUT/'evaluation_manifest.json').write_text(json.dumps(dict(models=provenance,validation_keys=VAL_KEYS,repo_commit=REPO_COMMIT,source=str(SOURCE),evaluation='Fresh inference on validation only; fixed pretrained detail targets',score_formula='0.5 * ((L+ab)/(baseline L+ab) + (1-IoU)/(1-baseline IoU))'),indent=2))
display(summary[['epoch','L_quantile','ab_quantile','target_mask_IoU','identity_cosine','protected_RGB_MAE_vs_input','detail_512_MAE_vs_pre','score']])


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
colors=['#7B8494','#3178C6','#098C79','#DB9031','#9966B3']
specs=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),('target_mask_IoU','Hair-mask IoU ↑'),('identity_cosine','Face identity cosine ↑'),('protected_RGB_MAE_vs_input','Protected-region MAE ↓'),('detail_512_MAE_vs_pre','Detail MAE vs baseline pre-output ↓'),('boundary_F1_2px','Hair boundary F1 ↑'),('protected_SSIM','Protected-region SSIM ↑'),('score','Combined color / shape score ↓')]
with PdfPages(OUT/'comparison_report.pdf') as pdf:
    # Each image uses the same case for all five models.
    for k in VAL_KEYS:
        fig,axes=plt.subplots(2,4,figsize=(16,9))
        names=[('face','Input face'),('aligned','Cached shape-aligned reference'),('ref','Color reference'),('Baseline','Pretrained baseline')]+[(t,f'Trial {t} | epoch {epochs[t]}') for t in 'ABCD']
        for ax,(name,label) in zip(axes.flat,names):
            with Image.open(OUT/'images'/k/(name+'.png')) as im:ax.imshow(im)
            ax.set_title(label,fontsize=11);ax.axis('off')
        fig.suptitle(k+' | Same cached inputs and masks',fontsize=17)
        fig.tight_layout();fig.savefig(OUT/(k+'_comparison.png'),dpi=160);pdf.savefig(fig)
        plt.show();plt.close(fig)
    fig,axes=plt.subplots(3,3,figsize=(15,12))
    for ax,(key,label) in zip(axes.flat,specs):
        values=summary[key].to_numpy()
        ax.bar(range(5),values,color=colors,width=.65)
        ax.set_xticks(range(5),['Baseline']+[f'{t}\nep {epochs[t]}' for t in 'ABCD'])
        ax.set_ylim(0,max(values)*1.2);ax.set_title(label,loc='left',fontsize=11)
        ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
        for i,v in enumerate(values):ax.text(i,v+max(values)*.025,f'{v:.4f}',ha='center',fontsize=9)
    fig.suptitle('Fresh evaluation | Baseline vs A–D | Validation mean, n=9',fontsize=18)
    fig.tight_layout(rect=(0,0,1,.96));fig.savefig(OUT/'metrics_comparison.png',dpi=180);pdf.savefig(fig)
    plt.show();plt.close(fig)
print('Completed:',OUT)
# This is a checkpoint evaluation, not a new epoch-by-epoch training history.


## Test: ชุดข้อมูล คู่ทดสอบและล็อก checkpoint

In [ ]:
# TEST: no optimizer, no backward, no checkpoint selection from test scores.
TEST_OUT=LOCAL_REPORT/'03_test';TEST_OUT.mkdir(exist_ok=True)
DATASET_ZIP=''  # Optional path to HairFastGAN_Dataset_Split_v2.zip in Drive; blank uploads the dataset.
from pathlib import PurePosixPath
if not DATASET_ZIP:
    print('Upload HairFastGAN_Dataset_Split_v2.zip containing the test images (not the review ZIP)')
    up=files.upload()
    if len(up)!=1:raise ValueError('Select one dataset ZIP')
    DATASET_ZIP=str(Path(next(iter(up))).resolve())
DATASET_ZIP=Path(DATASET_ZIP)
assert sha(DATASET_ZIP)=='5eb3a720c71989e6e36e1a38fae13840ec9e9853b5196893e190db499aa5719a','Dataset ZIP differs from reviewed v2'
DATA_DIR=Path('/content')/('test_data_'+sha(DATASET_ZIP)[:12]);DATA_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    if sum(i.file_size for i in z.infolist())>600*1024**2:raise ValueError('Unexpected dataset size')
    for info in z.infolist():
        p=PurePosixPath(info.filename)
        if p.is_absolute() or '..' in p.parts or ((info.external_attr>>16)&0o170000)==0o120000:raise ValueError('Unsafe ZIP entry')
    z.extractall(DATA_DIR)
mp=list(DATA_DIR.rglob('manifest.json'))
assert len(mp)==1
DATA_ROOT=mp[0].parent
DATA_META=json.loads(mp[0].read_text());ITEMS=DATA_META['items'];ITEM_BY_ID={r['id']:r for r in ITEMS}
for item in ITEMS:
    p=DATA_ROOT/item['file']
    assert p.resolve().is_relative_to(DATA_ROOT.resolve())
    assert sha(p)==item['sha256'],item['id']
test_items=json.loads((DATA_ROOT/'test_candidate.json').read_text())
assert len(test_items)==20
for item in test_items:assert ITEM_BY_ID[item['id']]['split']=='test_candidate'
trainval_cases=json.loads((SOURCE/'cases.json').read_text())
def resolve_dataset_id(value,role):
    category={'face':'faces','shape':'styles','color':'colors'}[role]
    token=str(value);stem=Path(token).stem
    found=[r['id'] for r in ITEMS if r['category']==category and (r['id']==token or r['id']==stem or r['file']==token or r.get('original_path')==token)]
    if len(found)!=1:raise ValueError(f'Cannot resolve historical {role}: {value}')
    return found[0]
used_ids={resolve_dataset_id(c[role],role) for c in trainval_cases if c['split'] in ['train','validation'] for role in ['face','shape','color']}
used_ids.update(r['id'] for r in ITEMS if r['split'] in ['train','validation'])
# Check across roles: same image bytes, decoded pixels, and declared identity/template groups.
used=[ITEM_BY_ID[k] for k in used_ids]
used_hashes={r['sha256'] for r in used};used_pixels={r.get('pixels') for r in used if r.get('pixels')}
used_groups={r['group_id'] for r in used}
for row in test_items:
    r=ITEM_BY_ID[row['id']]
    assert r['id'] not in used_ids and r['sha256'] not in used_hashes,('overlap',r['id'])
    assert r.get('pixels') not in used_pixels,('pixel overlap',r['id'])
    assert r['group_id'] not in used_groups,('group overlap',r['id'])
faces=sorted([r['id'] for r in test_items if r['category']=='faces'])
styles=sorted([r['id'] for r in test_items if r['category']=='styles'])
colors=sorted([r['id'] for r in test_items if r['category']=='colors'])
assert (len(faces),len(styles),len(colors))==(10,5,5)
rng=random.Random(3407);rng.shuffle(styles);rng.shuffle(colors)
TEST_CASES=[dict(key=f'test_{i:03d}',face=f,shape=styles[i%5],color=colors[i%5],face_group=ITEM_BY_ID[f]['group_id']) for i,f in enumerate(faces)]
(TEST_OUT/'test_pairs.json').write_text(json.dumps(TEST_CASES,indent=2))
TEST_KEYS=[r['key'] for r in TEST_CASES]
# Freeze validation-selected weights before generating any test output.
LOCK_DIR=Path('/content')/('locked_ABCD_'+uuid.uuid4().hex[:8]);LOCK_DIR.mkdir()
LOCK={'Baseline':{'blending_sha256':sha(b.opts.blending_checkpoint),'postprocess_sha256':sha(b.opts.pp_checkpoint),'epoch':0}}
for t in 'ABCD':
    original=STUDY_ROOT/f'trial_{t}/best.pt';snap=LOCK_DIR/f'{t}.pt'
    expected_hash=provenance[t]['sha256']
    assert sha(original)==expected_hash,f'{t}: checkpoint changed since validation evaluation'
    shutil.copy2(original,snap);assert sha(snap)==expected_hash
    ck=torch.load(snap,map_location='cpu',weights_only=True)
    assert int(ck['epoch'])==int(MANIFESTS[t]['best_epoch'])
    LOCK[t]=dict(epoch=int(ck['epoch']),sha256=expected_hash,original=str(original),snapshot=str(snap))
    del ck
(TEST_OUT/'checkpoint_lock.json').write_text(json.dumps(LOCK,indent=2))
TEST_PROTOCOL=dict(split='test_candidate',pairs=TEST_CASES,seed=3407,shape_rotation=False,color_rotation=True,mixing=.95,
    mask_policy='fixed automatic eroded hair masks; not manually reviewed validation masks',
    overlap_check='IDs, SHA256, manifest pixel hashes and declared groups vs current train/validation passed',
    historical_overlap='not verified against every older development/training dataset; candidate test results',
    face_group_count=len({r['face_group'] for r in TEST_CASES}),selection='checkpoints fixed by validation; no test-based selection',
    pairing='10 faces once each; shuffled 5 shapes and 5 colors repeated twice, seed 3407; frozen before outputs',repo_commit=REPO_COMMIT)
(TEST_OUT/'test_protocol.json').write_text(json.dumps(TEST_PROTOCOL,indent=2))
display(pd.DataFrame(TEST_CASES))
print('Candidate test split: 10 cases; older-development overlap remains unverified. No test-based tuning.')



## Test: ฟังก์ชันเตรียมภาพและ masks

In [ ]:
# Frozen preprocessing. Reuse official SEAN decode/e4e; do not use the custom trainable decoder.
from utils.shape_predictor import align_face
from models.encoder4editing.utils.model_utils import get_latents
from models.sean_codes.models.pix2pix_model import encode_sean,decode_sean
from models.Alignment import get_hair_face_code,get_new_shape
TEST_CACHE=Path('/content')/('test_cache_'+uuid.uuid4().hex[:8]);TEST_CACHE.mkdir()
e4e=hair_fast.embed.e4e
def erode_test(m,k=7):return -F.max_pool2d(-m.float(),k,1,k//2)
def dilate_test(m,k=11):return F.max_pool2d(m.float(),k,1,k//2)
@torch.no_grad()
def test_embedding(ident):
    path=TEST_CACHE/(ident+'.pt')
    if path.exists():return gpu(torch.load(path,map_location='cpu',weights_only=True))
    with Image.open(DATA_ROOT/ITEM_BY_ID[ident]['file']) as im:original=im.convert('RGB').copy()
    image=align_face([original])[0].unsqueeze(0).cuda();emb=hair_fast.embed
    i512=emb.downsample_512(image);i256=emb.downsample_256(image)
    W=get_latents(e4e,emb.normalize(i256))
    output=emb.encoder.test(img=emb.normalize(image),return_latent=True)
    feature16=output[-1];S=output[-2]
    feature32=G([S],input_is_latent=True,return_latents=False,start_layer=3,end_layer=3,layer_in=feature16,randomize_noise=False)[0]
    mask=segment(emb.to_bisenet(i512))
    h=F.interpolate((mask==13).float(),(32,32),mode='bicubic')
    fw=G([W],input_is_latent=True,return_latents=False,start_layer=0,end_layer=3,randomize_noise=False)[0]
    feature32=feature32+.95*h*(fw-feature32)
    result=cpu(dict(W=W,F=feature32,S=S,mask=mask,image_256=i256,image_norm_256=emb.normalize(i256)))
    torch.save(result,path);return gpu(result)

@torch.no_grad()
def prepare_test_case(row):
    key=row['key']
    seed=(3407+int(hashlib.sha256(key.encode()).hexdigest()[:8],16))%(2**31)
    torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    ef,es,ec=[test_embedding(row[r]) for r in ['face','shape','color']]
    face_code,_=get_hair_face_code(align.mask_generator,ef['mask'][0,0])
    _,hair_code=get_hair_face_code(align.mask_generator,es['mask'][0,0])
    labels=get_new_shape(align.mask_generator,face_code,hair_code)[None,None].long()
    target=labels==13;hf=(ef['mask']==13).float();hs=(es['mask']==13).float()
    codes=encode_sean(align.sean_model,torch.cat([ef['image_256'],es['image_256']],0),torch.cat([ef['mask'],es['mask']],0))
    gen1=decode_sean(align.sean_model,codes[0:1],labels)
    gen2=decode_sean(align.sean_model,codes[1:2],labels)
    aligned_embed=align.latent_encoder([gen1,gen2])
    a=aligned_embed['F'][0:1];c=aligned_embed['F'][1:2]
    masks=torch.cat([1-(1-hf)*(1-target.float()),target.float(),hs*target.float()],0)
    dil,ero=align.dilate_erosion.mask(masks)
    low=1-F.interpolate(torch.stack([dil[0],ero[1],ero[2]],0).float(),(32,32),mode='bicubic')
    feature=a+low[0]*(ef['F']-a);feature=c+low[1]*(feature-c);feature=es['F']+low[2]*(feature-es['F'])
    color_align=align.shape_module('face','color',{'face':ef,'color':ec})
    de=b.dilate_erosion.hair_from_mask(torch.cat([ef['mask'],ec['mask']],0))
    h1d=de[0][0].unsqueeze(0);h3d=de[0][1].unsqueeze(0);h3e=de[1][1].unsqueeze(0)
    hxd,_=b.dilate_erosion.mask(color_align['HM_X'])
    preserve=(1-h1d)*(1-h3d)*(1-hxd)
    protected=dilate_test(hf.bool()|target)<.5
    core=(erode_test(target)>.5)&~protected;refmask=erode_test(ec['mask']==13)>.5
    if min(int(core.sum()),int(refmask.sum()),int(protected.sum()))<100:raise ValueError('Insufficient mask pixels: '+key)
    aligned=G([ef['S']],input_is_latent=True,return_latents=False,start_layer=4,end_layer=8,layer_in=feature,randomize_noise=False)[0]
    d=dict(S=ef['S'],F=feature,I1=ef['image_norm_256'],face=ef['image_256'],shape=es['image_256'],ref=ec['image_256'],aligned=unit(aligned),target=target,protected=protected,core=core,refmask=refmask,eval_core=core,eval_refmask=refmask,
        args=[ef['S'][:,6:],ec['S'][:,6:],ef['image_norm_256']*preserve,ec['image_norm_256']*h3e])
    torch.save(cpu(d),TEST_CACHE/(key+'.pt'))



## Test: เตรียม cache คงที่สำหรับทั้ง 5 โมเดล

In [ ]:
reset_pretrained()
for row in TEST_CASES:
    prepare_test_case(row);print('Prepared test',row['key'])
# A preparation failure stops the run; no silent deletion or replacement of test cases.
TEST_FIXED={}
with torch.no_grad():
    for row in TEST_CASES:
        key=row['key'];d=gpu(torch.load(TEST_CACHE/(key+'.pt'),map_location='cpu',weights_only=True))
        pre,raw,s,sf,ff=joint_forward(d)
        TEST_FIXED[key]=cpu(dict(pre=unit(pre),pre_512=to_size(((pre+1)/2).clamp(0,1),512),s=s,sf=sf,ff=ff,face_embedding=id_features(d['face'])))
        folder=TEST_OUT/'images'/key;folder.mkdir(parents=True,exist_ok=True)
        for name in ['face','shape','ref','aligned']:saveim(d[name],folder/(name+'.png'))
        for name in ['target','protected','eval_core','eval_refmask']:
            Image.fromarray((d[name][0,0].cpu().numpy()*255).astype('uint8')).save(folder/(name+'.png'))
        del d,pre,raw,s,sf,ff


## Test: รัน baseline และ A–D โดยไม่ฝึก

In [ ]:
TEST_ROWS=[]
try:
    with torch.no_grad():
        for model in ['Baseline','A','B','C','D']:
            assert sha(b.opts.blending_checkpoint)==LOCK['Baseline']['blending_sha256']
            assert sha(b.opts.pp_checkpoint)==LOCK['Baseline']['postprocess_sha256']
            reset_pretrained()
            if model!='Baseline':
                path=Path(LOCK[model]['snapshot']);assert sha(path)==LOCK[model]['sha256']
                ck=torch.load(path,map_location='cpu',weights_only=True)
                state=enc.state_dict();state.update(ck['blending_trainable_state_dict']);enc.load_state_dict(state,strict=True)
                pp.load_state_dict(ck['post_process_state_dict'],strict=True);del state,ck
            for row in TEST_CASES:
                key=row['key'];d=gpu(torch.load(TEST_CACHE/(key+'.pt'),map_location='cpu',weights_only=True));fx=gpu(TEST_FIXED[key])
                pre,raw,s,sf,ff=joint_forward(d);_,values=objective(pre,raw,s,sf,ff,d,fx)
                im=unit(raw);values.update(shape_metrics(im,d));values.update(extra_metrics(im,d))
                numeric={k:float(v) for k,v in values.items()}
                assert all(np.isfinite(v) for v in numeric.values()),(model,key)
                TEST_ROWS.append(dict(model=model,epoch=LOCK[model]['epoch'],case=key,split='test_candidate',face_group=row['face_group'],**numeric))
                saveim((raw+1)/2,TEST_OUT/'images'/key/(model+'.png'))
                pd.DataFrame(TEST_ROWS).to_csv(TEST_OUT/'per_case_metrics.csv',index=False)
                del d,fx,pre,raw,s,sf,ff,values,im
                print('Test',model,key,'done')
finally:
    reset_pretrained()


## Test: ตาราง กราฟและภาพเปรียบเทียบ

In [ ]:
TEST_FRAME=pd.DataFrame(TEST_ROWS)
assert len(TEST_FRAME)==5*len(TEST_CASES) and not TEST_FRAME.duplicated(['model','case']).any()
ORDER=['Baseline','A','B','C','D']
TEST_MEAN=TEST_FRAME.groupby('model').mean(numeric_only=True).reindex(ORDER)
TEST_SD=TEST_FRAME.groupby('model').std(numeric_only=True,ddof=1).reindex(ORDER)
tb=TEST_MEAN.loc['Baseline']
TEST_MEAN['score']=.5*((TEST_MEAN.L_quantile+TEST_MEAN.ab_quantile)/max(tb.L_quantile+tb.ab_quantile,1e-8)+(1-TEST_MEAN.target_mask_IoU)/max(1-tb.target_mask_IoU,1e-8))
TEST_MEAN.to_csv(TEST_OUT/'mean_metrics.csv');TEST_SD.to_csv(TEST_OUT/'sd_across_cases.csv')
TEST_FRAME.groupby(['model','face_group']).mean(numeric_only=True).to_csv(TEST_OUT/'mean_by_face_group.csv')
# The score is a ratio of aggregate means; no misleading per-case SD is assigned to it.
cols=['L_quantile','ab_quantile','target_mask_IoU','identity_cosine','protected_RGB_MAE_vs_input','detail_512_MAE_vs_pre']
pd.DataFrame([dict(model=m,metric=k,mean=TEST_MEAN.loc[m,k],sd_cases=TEST_SD.loc[m,k],n_cases=len(TEST_CASES)) for m in ORDER for k in cols]).to_csv(TEST_OUT/'mean_sd_long.csv',index=False)
display(TEST_MEAN[['epoch']+cols+['score']])
print('All five models evaluated. SD describes spread across cases; repeated identities are not independent samples. No test winner is selected.')
from matplotlib.backends.backend_pdf import PdfPages
with PdfPages(TEST_OUT/'test_report.pdf') as pdf:
    for row in TEST_CASES:
        key=row['key'];fig,axs=plt.subplots(2,4,figsize=(16,9))
        panels=[('face','Input'),('shape','Shape reference'),('ref','Color reference'),('Baseline','Baseline')]+[(t,f'{t} | epoch {LOCK[t]["epoch"]}') for t in 'ABCD']
        for ax,(name,label) in zip(axs.flat,panels):
            with Image.open(TEST_OUT/'images'/key/(name+'.png')) as im:ax.imshow(im)
            ax.set_title(label);ax.axis('off')
        fig.suptitle(key+' | fixed test-candidate pair');fig.tight_layout();pdf.savefig(fig);fig.savefig(TEST_OUT/(key+'_comparison.png'),dpi=150);plt.show();plt.close(fig)
    fig,axs=plt.subplots(2,3,figsize=(15,9))
    labels=['Lightness error L ↓','Color error ab ↓','Hair-mask IoU ↑','Face identity cosine ↑','Protected MAE ↓','Detail MAE vs baseline pre-output ↓']
    for ax,k,label in zip(axs.flat,cols,labels):
        y=TEST_MEAN[k].to_numpy();e=TEST_SD[k].to_numpy()
        ax.errorbar(range(5),y,yerr=e,fmt='none',color='#667085',capsize=4)
        ax.scatter(range(5),y,c=['#7B8494','#3178C6','#098C79','#DB9031','#9966B3'],s=70,zorder=4)
        ax.set_xticks(range(5),ORDER);ax.set_title(label,loc='left');ax.grid(alpha=.2)
    fig.suptitle('Test candidate | Mean ± SD across 10 cases (not confidence intervals)')
    fig.tight_layout();pdf.savefig(fig);fig.savefig(TEST_OUT/'test_metrics_mean_sd.png',dpi=180);plt.show();plt.close(fig)
(TEST_OUT/'completion.json').write_text(json.dumps(dict(status='complete',models=ORDER,cases=len(TEST_CASES),metric_rows=len(TEST_ROWS),no_test_based_selection=True),indent=2))
print('Test results:',TEST_OUT)


## Barbershop-FT — ตั้งค่า source และ pretrained

In [ ]:
# Barbershop stages use the SAME Python/Torch/CUDA runtime as HairFastGAN via subprocess isolation.
BARBER_REV='088d7d0f0623b348732fc15d8fbdea8b8cb730c7'
BARBER_REPO=Path('/content/Barbershop_matched_experiment')
BARBER_ROOT=LOCAL_REPORT/'04_barbershop_FT_setup';BARBER_ROOT.mkdir(exist_ok=True)
BARBER_WEIGHTS=Path('/content/drive/MyDrive/Barbershop_matched_weights');BARBER_WEIGHTS.mkdir(exist_ok=True)
if not (BARBER_REPO/'.git').exists():subprocess.run(['git','clone','https://github.com/ZPdesu/Barbershop.git',str(BARBER_REPO)],check=True)
# No destructive checkout of unrelated work: this is a dedicated experiment checkout.
head=subprocess.check_output(['git','-C',str(BARBER_REPO),'rev-parse','HEAD'],text=True).strip()
if head!=BARBER_REV:
    assert not subprocess.check_output(['git','-C',str(BARBER_REPO),'status','--porcelain'],text=True).strip(), 'Dedicated checkout has modifications at another commit'
    subprocess.run(['git','-C',str(BARBER_REPO),'checkout',BARBER_REV],check=True)
import gdown, ast, platform
# Download once; use the SAME source weights through a local/Drive path or manual upload on failure.
# Optional: fill paths to the original Barbershop weights from your earlier notebook.
LOCAL_WEIGHT_PATHS = {'ffhq.pt': '', 'seg.pth': ''}
WEIGHT_IDS = {'ffhq.pt': '1NVlMTYO4wyXZYVGVlGjP_xoCzBlfXSR1',
              'seg.pth': '1bDOOJpST8dJDcSPuPKpTbdAmLKUGUji5'}

def check_barber_weight(path, name):
    if not path.is_file() or path.stat().st_size < 1000000:
        raise ValueError('Missing/incomplete weight file: '+str(path))
    # Only open original weights from the sources above or your trusted existing Barbershop copy.
    obj=torch.load(path,map_location='cpu',weights_only=False)
    if name=='ffhq.pt':
        if not isinstance(obj,dict) or not {'g_ema','latent_avg'}<=set(obj):
            raise ValueError('Not a Barbershop ffhq.pt checkpoint (requires g_ema and latent_avg)')
        if not isinstance(obj['g_ema'],dict) or not torch.is_tensor(obj['latent_avg']):
            raise ValueError('Invalid generator checkpoint structure')
    elif not isinstance(obj,dict) or not any(torch.is_tensor(v) for v in obj.values()):
        raise ValueError('Expected original Barbershop segmentation state_dict')
    del obj

for name,drive_id in WEIGHT_IDS.items():
    target=BARBER_WEIGHTS/name
    if target.is_file():
        check_barber_weight(target,name)
        print('Using cached weight:',target)
        continue
    temp=BARBER_WEIGHTS/(name+'.download')
    supplied=LOCAL_WEIGHT_PATHS[name].strip()
    if supplied:
        source=Path(supplied).expanduser()
        check_barber_weight(source,name)
        shutil.copy2(source,temp)
    else:
        try:
            result=gdown.download(id=drive_id,output=str(temp),quiet=False)
            if result is None:raise RuntimeError('gdown returned no file')
            check_barber_weight(temp,name)
        except Exception as err:
            print('Automatic download unavailable for',name,':',type(err).__name__)
            print('Original source: https://drive.google.com/file/d/'+drive_id+'/view')
            print('Use this original file or your existing Barbershop copy. Do not substitute another model checkpoint.')
            supplied=input('Paste the full /content/drive/... path to '+name+'; or press Enter to upload it: ').strip().strip('"')
            if supplied:
                source=Path(supplied).expanduser();check_barber_weight(source,name)
                shutil.copy2(source,temp)
            else:
                from google.colab import files as colab_files
                # Upload into a dedicated folder; avoid overwriting unrelated notebook files.
                import tempfile
                with tempfile.TemporaryDirectory(prefix='barber_weight_') as upload_dir:
                    before=os.getcwd()
                    try:
                        os.chdir(upload_dir);uploaded_weights=colab_files.upload()
                    finally:os.chdir(before)
                    if len(uploaded_weights)!=1:raise ValueError('Upload only '+name+' in this dialog')
                    source=Path(upload_dir)/next(iter(uploaded_weights))
                    check_barber_weight(source,name);shutil.copy2(source,temp)
                    del uploaded_weights
    check_barber_weight(temp,name)
    os.replace(temp,target)
    print('Saved validated weight:',target)
# Record byte identities; structural validation alone does not establish source provenance.
weight_hashes={name:sha(BARBER_WEIGHTS/name) for name in WEIGHT_IDS}
weight_lock=BARBER_WEIGHTS/'weight_hashes.json'
if weight_lock.exists():
    assert json.loads(weight_lock.read_text())==weight_hashes, 'Cached weights changed; do not mix checkpoints within an experiment'
else:weight_lock.write_text(json.dumps(weight_hashes,indent=2))

# Use the repository's fixed PCA; do not rebuild a new random PCA each trial.
pca=BARBER_WEIGHTS/'ffhq_PCA.npz'
if not pca.is_file():shutil.copy2(BARBER_REPO/'pretrained_models/ffhq_PCA.npz',pca)
assert sha(pca)==sha(BARBER_REPO/'pretrained_models/ffhq_PCA.npz')
weights=torch.load(BARBER_WEIGHTS/'ffhq.pt',map_location='cpu',weights_only=False)
assert {'g_ema','latent_avg'}<=set(weights);del weights

# Use the same native-PyTorch operator implementation used by HairFastGAN.
patches=[]
for kind in ['fused_act','upfirdn2d']:
    path=BARBER_REPO/'models/stylegan2/op'/(kind+'.py')
    original=subprocess.check_output(['git','-C',str(BARBER_REPO),'show',BARBER_REV+':models/stylegan2/op/'+kind+'.py'],text=True)
    patched=_native_stylegan_source(original,kind)
    compile(patched,str(path),'exec');path.write_text(patched)
    patches.append(dict(file=str(path.relative_to(BARBER_REPO)),sha256=sha(path)))

# Instrument existing optimizer steps without changing losses or latent update rules.
class LogOptimizerSteps(ast.NodeTransformer):
    def __init__(self,stage):self.stage=stage;self.counter=0
    def visit_Expr(self,node):
        self.generic_visit(node)
        call=node.value
        if isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute) and call.func.attr=='step' and isinstance(call.func.value,ast.Name) and call.func.value.id in ['optimizer_W','optimizer_FS','optimizer_align','opt_blend']:
            self.counter+=1
            log=ast.parse(f"record('{self.stage}_{self.counter}', locals().get('step'), locals().get('loss'))").body[0]
            return [node,ast.copy_location(log,node)]
        return node
for name,expected in [('Embedding',2),('Alignment',4),('Blending',1)]:
    rel='models/'+name+'.py';original=subprocess.check_output(['git','-C',str(BARBER_REPO),'show',BARBER_REV+':'+rel],text=True)
    transformer=LogOptimizerSteps(name);tree=transformer.visit(ast.parse(original));assert transformer.counter==expected,(name,transformer.counter)
    tree.body.insert(0,ast.ImportFrom(module='experiment_trace',names=[ast.alias(name='record')],level=0))
    ast.fix_missing_locations(tree);code=ast.unparse(tree);compile(code,rel,'exec');(BARBER_REPO/rel).write_text(code)
    patches.append(dict(file=rel,sha256=sha(BARBER_REPO/rel)))

(BARBER_REPO/'experiment_worker.py').write_text('"""Runs only inside the pinned Barbershop checkout in a separate Python process."""\nimport os,sys,json,time,random,hashlib,copy,gc,ast,argparse\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom PIL import Image\nif not hasattr(Image,\'ANTIALIAS\'):Image.ANTIALIAS=Image.Resampling.LANCZOS\ncfg=json.loads(Path(sys.argv[1]).read_text())\n# Read the original CLI defaults without executing its main pipeline.\nsource=Path(\'main.py\').read_text();tree=ast.parse(source)\nblock=next(n for n in tree.body if isinstance(n,ast.If) and \'__name__\' in ast.unparse(n.test))\nparser_nodes=[]\nfor node in block.body:\n    if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'args\' for t in node.targets):break\n    parser_nodes.append(node)\nns={\'argparse\':argparse};exec(compile(ast.Module(body=parser_nodes,type_ignores=[]),\'defaults\',\'exec\'),ns)\nopts=ns[\'parser\'].parse_args([])\nopts.ckpt=cfg[\'ckpt\'];opts.seg_ckpt=cfg[\'seg\'];opts.seed=int(cfg.get(\'seed\',3407))\nopts.W_steps=1100;opts.FS_steps=250;opts.align_steps1=140;opts.align_steps2=100;opts.blend_steps=400\nopts.save_intermediate=False;opts.verbose=False;opts.sign=\'realistic\';opts.smooth=5\n# All torch.load calls are restricted to the trusted local model files and repository LPIPS weights.\n_original_load=torch.load\ndef legacy_load(*args,**kwargs):\n    kwargs.setdefault(\'weights_only\',False)\n    return _original_load(*args,**kwargs)\ntorch.load=legacy_load\nfrom models.Embedding import Embedding\nfrom models.Alignment import Alignment\nfrom models.Blending import Blending\nimport experiment_trace as trace\n\ndef seed(key):\n    n=(int(cfg.get(\'seed\',3407))+int(hashlib.sha256(key.encode()).hexdigest()[:8],16))%(2**31)\n    random.seed(n);np.random.seed(n);torch.manual_seed(n);torch.cuda.manual_seed_all(n)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for chunk in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\ndef fingerprint(module):\n    h=hashlib.sha256()\n    for k,v in module.state_dict().items():h.update(k.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\ndef atomic(obj,p):\n    p=Path(p);tmp=p.with_suffix(\'.tmp\');tmp.write_text(json.dumps(obj,indent=2));os.replace(tmp,p)\ndef seconds(fn):\n    torch.cuda.synchronize();tick=time.perf_counter();result=fn();torch.cuda.synchronize()\n    return time.perf_counter()-tick,result\n\nroot=Path(cfg[\'root\']);inv=root/\'inversion\';inv.mkdir(exist_ok=True)\ntrace.configure(root/\'optimization_steps.csv\')\nseed(\'initialization\')\nopts.learning_rate=.01;opts.output_dir=str(inv)\nembedding=Embedding(copy.deepcopy(opts))\n# Use fixed pretrained noise in both frozen preprocessing and subsequent shared-weight decoding.\nimport functools\nembedding.net.generator.forward=functools.partial(embedding.net.generator.forward,randomize_noise=False)\ninitial=fingerprint(embedding.net.generator)\n# Reuse the exact same generator for all modules; these weights remain frozen.\naligner=Alignment(copy.deepcopy(opts),net=embedding.net)\nblender=Blending(copy.deepcopy(opts),net=embedding.net)\nfor module in [embedding.loss_builder,aligner.loss_builder,blender.loss_builder]:\n    module.eval().requires_grad_(False)\n# No GPU training of shared network weights occurs.\natomic(dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),gpu=torch.cuda.get_device_name(0),defaults=vars(opts),weights_sha256=sha(cfg[\'ckpt\']),seg_sha256=sha(cfg[\'seg\']),pca_sha256=sha(cfg[\'pca\'])),root/\'worker_environment.json\')\nids=sorted({r[k] for r in cfg[\'cases\'] for k in [\'face\',\'shape\',\'color\']})\nfor ident in ids:\n    marker=inv/(ident+\'_complete.json\');image=str(Path(cfg[\'input_dir\'])/(ident+\'.png\'))\n    if marker.exists():\n        previous=json.loads(marker.read_text());assert previous[\'input_sha256\']==sha(image)\n        continue\n    seed(\'inversion/\'+ident);trace.context(model=\'shared_inversion\',case=ident,operation=\'inversion\')\n    wsec,_=seconds(lambda:embedding.invert_images_in_W([image]))\n    fsec,_=seconds(lambda:embedding.invert_images_in_FS([image]))\n    trace.flush()\n    atomic(dict(input_sha256=sha(image),W_seconds=wsec,FS_seconds=fsec,total_seconds=wsec+fsec,W_steps=opts.W_steps,FS_steps=opts.FS_steps),marker)\n\nrows=[]\nfor spec in cfg[\'trials\']:\n    for row in cfg[\'cases\']:\n        key=row[\'key\'];dest=root/spec[\'name\']/key;dest.mkdir(parents=True,exist_ok=True)\n        result_path=dest/\'complete.json\'\n        if result_path.exists():\n            old=json.loads(result_path.read_text());assert old[\'spec\']==spec\n            assert Path(old[\'final_float\']).is_file() and Path(old[\'final_png\']).is_file(), \'Completed result files missing\'\n            rows.append(old);continue\n        # Shared inversion bytes are read-only and linked into each isolated case directory.\n        for sub in [\'W+\',\'FS\']:\n            link=dest/sub\n            if not link.exists():link.symlink_to(inv/sub,target_is_directory=True)\n        a=copy.deepcopy(opts);a.output_dir=str(dest);a.learning_rate=spec[\'align_lr\'];aligner.opts=a;aligner.loss_builder.opt=a\n        b=copy.deepcopy(opts);b.output_dir=str(dest);b.learning_rate=spec[\'blend_lr\'];blender.opts=b;blender.loss_builder.opt=b\n        f,s,c=[str(Path(cfg[\'input_dir\'])/(row[k]+\'.png\')) for k in [\'face\',\'shape\',\'color\']]\n        seed(\'case/\'+key) # same latent/noise initialization for all trials of this case\n        trace.context(model=spec[\'name\'],case=key,operation=\'align_shape\')\n        align1,_=seconds(lambda:aligner.align_images(f,s,sign=\'realistic\',align_more_region=False,smooth=5))\n        trace.context(model=spec[\'name\'],case=key,operation=\'align_color\')\n        align2=0.0\n        if s!=c:\n            align2,_=seconds(lambda:aligner.align_images(f,c,sign=\'realistic\',align_more_region=False,smooth=5,save_intermediate=False))\n        trace.context(model=spec[\'name\'],case=key,operation=\'blend\')\n        # Capture the float output before PNG quantization for the common evaluator.\n        capture={};original_save=blender.save_blend_results\n        def save_float(*args,**kwargs):\n            capture[\'full\']=((args[4].detach().float().cpu()+1)/2).clamp(0,1).numpy()\n            return original_save(*args,**kwargs)\n        blender.save_blend_results=save_float\n        try:blendsec,_=seconds(lambda:blender.blend_images(f,s,c,sign=\'realistic\'))\n        finally:blender.save_blend_results=original_save\n        np.save(dest/\'final_float.npy\',capture[\'full\']);del capture\n        from PIL import Image\n        png=dest/f\'{Path(f).stem}_{Path(s).stem}_{Path(c).stem}_realistic.png\'\n        assert png.is_file()\n        inv_seconds=sum(json.loads((inv/(ident+\'_complete.json\')).read_text())[\'total_seconds\'] for ident in sorted({row[k] for k in [\'face\',\'shape\',\'color\']}))\n        result=dict(case=key,split=cfg[\'split\'],model=spec[\'name\'],spec=spec,align_shape_seconds=align1,align_color_seconds=align2,blend_seconds=blendsec,inversion_cold_seconds=inv_seconds,cached_pipeline_seconds=align1+align2+blendsec,estimated_cold_compute_seconds=inv_seconds+align1+align2+blendsec,final_float=str(dest/\'final_float.npy\'),final_png=str(png))\n        trace.flush()\n        atomic(result,result_path);rows.append(result)\n        atomic(rows,root/\'results.json\');gc.collect();torch.cuda.empty_cache()\nassert fingerprint(embedding.net.generator)==initial,\'Frozen generator changed\'\natomic(rows,root/\'results.json\')\natomic(dict(status=\'complete\',frozen_generator_unchanged=True,cases=len(cfg[\'cases\']),trials=len(cfg[\'trials\']),rows=len(rows)),root/\'completion.json\')\n')
(BARBER_REPO/'experiment_trace.py').write_text("import csv,torch,time\n_handle=None\n_writer=None\n_context={}\ndef configure(path):\n    global _handle,_writer\n    fields=['model','case','operation','stage','step','loss']\n    # Keep only completed work when retrying; partial cases restart from the shared inversion.\n    if path.exists():\n        with open(path,newline='') as src:rows=list(csv.DictReader(src))\n        def complete(r):\n            marker=(path.parent/'inversion'/(r['case']+'_complete.json')) if r['model']=='shared_inversion' else (path.parent/r['model']/r['case']/'complete.json')\n            return marker.is_file()\n        temp=path.with_suffix('.tmp')\n        with open(temp,'w',newline='') as dst:\n            writer=csv.DictWriter(dst,fieldnames=fields);writer.writeheader();writer.writerows(r for r in rows if complete(r))\n        temp.replace(path)\n    exists=path.exists() and path.stat().st_size>0\n    _handle=open(path,'a',newline='')\n    _writer=csv.DictWriter(_handle,fieldnames=fields)\n    if not exists:_writer.writeheader()\ndef flush():\n    if _handle:_handle.flush()\ndef context(**kwargs):\n    global _context\n    _context=kwargs\n    flush()\ndef record(stage,step,loss):\n    if _writer is None:return\n    value=float(loss.detach()) if torch.is_tensor(loss) else float(loss)\n    if not __import__('math').isfinite(value):raise RuntimeError('Nonfinite optimization loss')\n    _writer.writerow(dict(**_context,stage=stage,step=step,loss=value))\n")


## ล็อก protocol และเตรียม train/validation latent
ขั้นนี้ยังไม่ฝึก shared weights แต่คำนวณ input latent คงที่เพื่อให้ทั้งสี่ trials เริ่มจากฐานเดียวกัน

In [ ]:
FT_IMPLEMENTATION_HASH='efe9d28eee54aeb57cdf91763973484e02a2e28a2df33e697c9c75475fd80da3'
FT_TRAIN_SOURCE='# Load only the Barbershop synthesis module under a private package name.\n# This avoids replacing HairFastGAN\'s \'models\' or \'utils\' modules.\nimport importlib\nprivate=Path(\'/content/bs_ft_private\');package=private/\'bs_ft_synthesis\';package.mkdir(parents=True,exist_ok=True)\n(package/\'__init__.py\').write_text(\'\')\nshutil.copytree(BARBER_REPO/\'models/stylegan2/op\',package/\'op\',dirs_exist_ok=True)\nmodel_source=(BARBER_REPO/\'models/stylegan2/model.py\').read_text().replace(\'from models.stylegan2.op import\',\'from .op import\')\n(package/\'model.py\').write_text(model_source)\nif str(private) not in sys.path:sys.path.insert(0,str(private))\nBSGenerator=importlib.import_module(\'bs_ft_synthesis.model\').Generator\nbs=BSGenerator(1024,512,8,channel_multiplier=2)\ninit=torch.load(BARBER_WEIGHTS/\'ffhq.pt\',map_location=\'cpu\',weights_only=False)[\'g_ema\']\nbs.load_state_dict(init,strict=True);del init\nbs.eval().requires_grad_(False)\nfor module in list(bs.convs)[6:]+list(bs.to_rgbs)[3:]:module.requires_grad_(True)\nNAMED=dict(bs.named_parameters());NAMES=[n for n,p in NAMED.items() if p.requires_grad]\nCONV_NAMES=[n for n in NAMES if n.startswith(\'convs.\')];RGB_NAMES=[n for n in NAMES if n.startswith(\'to_rgbs.\')]\nassert CONV_NAMES and RGB_NAMES and set(NAMES)==set(CONV_NAMES+RGB_NAMES)\nINITIAL={n:NAMED[n].detach().cpu().clone() for n in NAMES}\ndef state_hash(values):\n    h=hashlib.sha256()\n    for n,v in sorted(values.items()):h.update(n.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\ndef frozen_hash():return state_hash({n:v for n,v in bs.state_dict().items() if n not in NAMES})\nFROZEN_HASH=frozen_hash();INITIAL_HASH=state_hash(INITIAL)\nwrite_json(dict(initial_trainable_sha256=INITIAL_HASH,frozen_sha256=FROZEN_HASH,\n    trainable_parameters=sum(NAMED[n].numel() for n in NAMES),names=NAMES,\n    python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),\n    fine_tuning=\'FP32; fixed noise; deterministic per-epoch shuffle; no claim of equal FLOPs to HairFastGAN\'),FT_ROOT/\'environment_and_parameters.json\')\nbs.cuda();id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)\ntorch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\ntorch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n\ndef load_trainable(values):\n    assert set(values)==set(NAMES)\n    with torch.no_grad():\n        for n in NAMES:NAMED[n].copy_(values[n].to(NAMED[n]))\ndef trainable_state():return {n:NAMED[n].detach().cpu().clone() for n in NAMES}\ndef bs_render(S,feat,grad=False):\n    # Equivalent to official generator start_layer=4/end_layer=8; checkpoint each resolution.\n    out=feat;skip=None\n    for j in range(3,len(bs.to_rgbs)):\n        i=1+2*j\n        def block(x,w,rgb,j=j,i=i):\n            x=bs.convs[2*j](x,w[:,i],noise=getattr(bs.noises,f\'noise_{1+2*j}\'))\n            x=bs.convs[2*j+1](x,w[:,i+1],noise=getattr(bs.noises,f\'noise_{2+2*j}\'))\n            return x,bs.to_rgbs[j](x,w[:,i+2],rgb)\n        out,skip=checkpoint(block,out,S,skip,use_reentrant=False) if grad else block(out,S,skip)\n    return skip\n\ndef case_data(key,split):\n    d=load_case(key) if split!=\'test_candidate\' else torch.load(TEST_CACHE/(key+\'.pt\'),map_location=\'cpu\',weights_only=True)\n    # Keep only common supervision tensors, not HairFastGAN latent variables.\n    return gpu({k:d[k] for k in [\'face\',\'ref\',\'target\',\'protected\',\'eval_core\',\'eval_refmask\']})\nFRONT_INDEX={}\ndef front_record(key,split):\n    if split not in FRONT_INDEX:\n        records=json.loads((BARBER_ROOT/split/\'results.json\').read_text())\n        FRONT_INDEX[split]={r[\'case\']:r for r in records}\n    return FRONT_INDEX[split][key]\ndef latent(key,split):\n    rec=front_record(key,split);folder=Path(rec[\'final_png\']).parent/\'Blend_realistic\'\n    paths=list(folder.glob(\'*.npz\'));assert len(paths)==1\n    with np.load(paths[0]) as z:S=torch.from_numpy(z[\'latent_in\'].copy()).cuda();feat=torch.from_numpy(z[\'latent_F\'].copy()).cuda()\n    return S.detach(),feat.detach()\ndef reference(key,split,d):\n    rec=front_record(key,split);full=torch.from_numpy(np.load(rec[\'final_float\'])).cuda()\n    with torch.no_grad():return dict(pre=to_size(full,256),pre_512=to_size(full,512),face_embedding=id_features(d[\'face\']))\n\ndef ft_objective(raw,d,fx):\n    full=((raw+1)/2).clamp(0,1);im=to_size(full,256)\n    L,ab=quantile_color(im,d[\'ref\'],d[\'eval_core\'],d[\'eval_refmask\'])\n    identity,shape,_,_=constraint_terms(im,d,fx)\n    protected=masked_mae(im,d[\'face\'],d[\'protected\']);details=detail_terms(full,d,fx)\n    texture=torch.stack(list(details.values())).mean();color=(L+ab)/100\n    w=FT_PROTOCOL[\'weights\']\n    loss=w[\'final_color\']*color+w[\'final_shape\']*shape+w[\'identity\']*identity+w[\'protected\']*protected+w[\'detail\']*texture\n    return loss,dict(loss=loss,L_quantile=L,ab_quantile=ab,shape_loss=shape,identity_cosine=1-identity,\n        protected_RGB_MAE_vs_input=protected,detail_512_MAE_vs_own_baseline=details[512])\n\n@torch.no_grad()\ndef evaluate_bs(keys,split,epoch,model,save_images=False):\n    rows=[]\n    for key in keys:\n        d=case_data(key,split);fx=reference(key,split,d);S,feat=latent(key,split)\n        raw=bs_render(S,feat);loss,vals=ft_objective(raw,d,fx)\n        im=to_size(((raw+1)/2).clamp(0,1),256)\n        vals={k:float(v) for k,v in vals.items()};vals.update(shape_metrics(im,d));vals.update(extra_metrics(im,d))\n        assert all(np.isfinite(v) for v in vals.values())\n        rows.append(dict(case=key,epoch=epoch,model=model,split=split,**vals))\n        if save_images:\n            folder=FT_REPORT/split/\'images\'/key;folder.mkdir(parents=True,exist_ok=True);saveim((raw+1)/2,folder/(model+\'.png\'))\n        del d,fx,S,feat,raw,im,loss\n    return rows\n\ndef summarize(rows):return pd.DataFrame(rows).mean(numeric_only=True).to_dict()\ndef selection(v,base):\n    score=.5*((v[\'L_quantile\']+v[\'ab_quantile\'])/max(base[\'L_quantile\']+base[\'ab_quantile\'],1e-8)+(1-v[\'target_mask_IoU\'])/max(1-base[\'target_mask_IoU\'],1e-8))\n    g=FT_PROTOCOL[\'gates\']\n    eligible=(v[\'L_quantile\']+v[\'ab_quantile\']<=base[\'L_quantile\']+base[\'ab_quantile\'] and\n        v[\'target_mask_IoU\']>=base[\'target_mask_IoU\'] and v[\'identity_cosine\']>=base[\'identity_cosine\']-g[\'identity_drop\'] and\n        v[\'protected_RGB_MAE_vs_input\']<=base[\'protected_RGB_MAE_vs_input\']+g[\'protected_increase\'] and\n        v[\'detail_512_MAE_vs_own_baseline\']<=base[\'detail_512_MAE_vs_own_baseline\']+g[\'detail_absolute_increase\'])\n    return float(score),bool(eligible)\n\ndef atomic_torch(value,path):\n    temp=Path(str(path)+\'.tmp\');torch.save(value,temp);os.replace(temp,path)\ndef checkpoint_payload(spec,epoch,weights):\n    return dict(format=\'barbershop_shared_synthesis_ft_v2\',protocol_id=FT_ID,trial=spec,epoch=epoch,\n        initial_sha256=INITIAL_HASH,pretrained_sha256=FT_PROTOCOL[\'ffhq_sha256\'],state_dict=weights)\n\ndef run_trial(spec):\n    root=FT_ROOT/(\'trial_\'+spec[\'name\']);root.mkdir(exist_ok=True)\n    load_trainable(INITIAL)\n    assert state_hash(trainable_state())==INITIAL_HASH\n    opt=torch.optim.Adam([{\'params\':[NAMED[n] for n in CONV_NAMES],\'lr\':spec[\'synthesis_lr\']},\n                          {\'params\':[NAMED[n] for n in RGB_NAMES],\'lr\':spec[\'rgb_lr\']}],betas=(.9,.999))\n    steps=[];history=[];best_epoch=0;best_score=1.;best_state=copy.deepcopy(INITIAL);start=1\n    last_path=root/\'last.pt\'\n    if last_path.exists():\n        ck=torch.load(last_path,map_location=\'cpu\',weights_only=True)\n        assert ck[\'protocol_id\']==FT_ID and ck[\'trial\']==spec and ck[\'initial_sha256\']==INITIAL_HASH\n        load_trainable(ck[\'state_dict\']);opt.load_state_dict(ck[\'optimizer\'])\n        for state in opt.state.values():\n            for k,v in state.items():\n                if torch.is_tensor(v) and k!=\'step\':state[k]=v.cuda()\n        steps=ck[\'steps\'];history=ck[\'history\'];best_epoch=ck[\'best_epoch\'];best_score=ck[\'best_score\'];best_state=ck[\'best_state\'];start=ck[\'epoch\']+1\n        baseline=ck[\'baseline\'];del ck\n        assert len(steps)==150*(start-1)\n    else:\n        history=evaluate_bs(VAL_KEYS,\'validation\',0,spec[\'name\']);baseline=summarize(history)\n    def persist(epoch):\n        # The atomic last checkpoint is the sole resume authority. Derived CSV/best files are rebuilt from it.\n        payload=checkpoint_payload(spec,epoch,trainable_state())\n        payload.update(optimizer=cpu(opt.state_dict()),steps=steps,history=history,baseline=baseline,best_state=best_state,best_epoch=best_epoch,best_score=best_score)\n        atomic_torch(payload,last_path)\n        atomic_torch(checkpoint_payload(spec,best_epoch,best_state),root/\'best.pt\')\n        pd.DataFrame(steps).to_csv(root/\'training_steps.csv\',index=False)\n        pd.DataFrame(history).to_csv(root/\'validation_per_case.csv\',index=False)\n        write_json(dict(status=\'complete\' if epoch==EPOCHS else \'running\',completed_epoch=epoch,weight_updates=len(steps),\n            best_epoch=best_epoch,best_score=best_score,trial=spec,initial_sha256=INITIAL_HASH,\n            current_sha256=state_hash(trainable_state()),protocol_id=FT_ID),root/\'manifest.json\')\n    persist(start-1)\n    for epoch in range(start,EPOCHS+1):\n        random.seed(SEED+epoch);np.random.seed(SEED+epoch);torch.manual_seed(SEED+epoch);torch.cuda.manual_seed_all(SEED+epoch)\n        order=TRAIN_KEYS.copy();random.Random(SEED+epoch).shuffle(order)\n        for key in order:\n            assert key in TRAIN_KEYS and key not in VAL_KEYS\n            d=case_data(key,\'train\');fx=reference(key,\'train\',d);S,feat=latent(key,\'train\')\n            opt.zero_grad(set_to_none=True);torch.cuda.synchronize();tick=time.perf_counter();torch.cuda.reset_peak_memory_stats()\n            raw=bs_render(S,feat,grad=True);loss,vals=ft_objective(raw,d,fx)\n            assert loss.requires_grad and torch.isfinite(loss)\n            loss.backward()\n            for names in [CONV_NAMES,RGB_NAMES]:\n                grads=[NAMED[n].grad for n in names if NAMED[n].grad is not None]\n                assert grads and all(torch.isfinite(v).all() for v in grads)\n                assert sum(float(v.square().sum()) for v in grads)>0\n            assert all(p.grad is None for n,p in NAMED.items() if n not in NAMES)\n            assert all(p.grad is None for m in [id_net,parser_net] for p in m.parameters())\n            norm=torch.nn.utils.clip_grad_norm_([NAMED[n] for n in NAMES],1.);assert torch.isfinite(norm)\n            opt.step()\n            with torch.no_grad():train_shape=shape_metrics(to_size(((raw.detach()+1)/2).clamp(0,1),256),d)\n            torch.cuda.synchronize()\n            steps.append(dict(epoch=epoch,case=key,update=len(steps)+1,seconds=time.perf_counter()-tick,\n                gradient_norm=float(norm),peak_vram_MB=torch.cuda.max_memory_allocated()/1024**2,\n                **{k:float(v.detach()) for k,v in vals.items()},**train_shape))\n            print(f\'BS-FT {spec["name"]} epoch {epoch}/30 | update {len(steps)}/4500 | loss {float(loss.detach()):.5f}\',flush=True)\n            del d,fx,S,feat,raw,loss,vals,grads\n        opt.zero_grad(set_to_none=True)\n        val=evaluate_bs(VAL_KEYS,\'validation\',epoch,spec[\'name\']);history.extend(val)\n        score,eligible=selection(summarize(val),baseline)\n        if eligible and score<best_score:best_score=score;best_epoch=epoch;best_state=trainable_state()\n        assert state_hash(trainable_state())!=INITIAL_HASH,\'Shared network weights did not change\'\n        assert frozen_hash()==FROZEN_HASH,\'Frozen parameters or buffers changed\'\n        persist(epoch)\n        print(\'Validation score:\',score,\'eligible:\',eligible,\'best epoch:\',best_epoch)\n    assert len(steps)==4500 and len(history)==31*9\n    # final is genuinely epoch30, distinct from best.\n    atomic_torch(checkpoint_payload(spec,30,trainable_state()),root/\'final_epoch30.pt\')\n    del opt;gc.collect();torch.cuda.empty_cache()\n    return dict(trial=spec[\'name\'],best_epoch=best_epoch,score=best_score,weight_updates=len(steps),checkpoint=str(root/\'best.pt\'))\n\n# Preflight on one TRAIN case: official/activation-checkpointed forward equality,\n# nonzero gradients in both groups, a real optimizer step, then exact reset.\nload_trainable(INITIAL)\nkey=TRAIN_KEYS[0];d=case_data(key,\'train\');fx=reference(key,\'train\',d);S,feat=latent(key,\'train\')\nwith torch.no_grad():\n    official=bs([S],input_is_latent=True,start_layer=4,end_layer=8,layer_in=feat,randomize_noise=False)[0]\n    rendered=bs_render(S,feat)\n    torch.testing.assert_close(rendered,official,rtol=1e-5,atol=1e-6)\n    expected=torch.from_numpy(np.load(front_record(key,\'train\')[\'final_float\'])).cuda()\n    torch.testing.assert_close(((rendered+1)/2).clamp(0,1),expected,rtol=1e-4,atol=1e-5)\nprobe_opt=torch.optim.Adam([{\'params\':[NAMED[n] for n in CONV_NAMES],\'lr\':FT_TRIALS[0][\'synthesis_lr\']},\n                            {\'params\':[NAMED[n] for n in RGB_NAMES],\'lr\':FT_TRIALS[0][\'rgb_lr\']}])\nraw=bs_render(S,feat,True);loss,vals=ft_objective(raw,d,fx);loss.backward()\nprobe={}\nfor label,names in [(\'synthesis\',CONV_NAMES),(\'rgb\',RGB_NAMES)]:\n    grads=[NAMED[n].grad for n in names if NAMED[n].grad is not None]\n    assert grads and all(torch.isfinite(v).all() for v in grads)\n    probe[label+\'_gradient_norm\']=sum(float(v.square().sum()) for v in grads)**.5\n    assert probe[label+\'_gradient_norm\']>0\nprobe_opt.step()\nfor label,names in [(\'synthesis\',CONV_NAMES),(\'rgb\',RGB_NAMES)]:\n    probe[label+\'_weight_delta\']=sum(float((NAMED[n].detach().cpu()-INITIAL[n]).square().sum()) for n in names)**.5\n    assert probe[label+\'_weight_delta\']>0\nassert frozen_hash()==FROZEN_HASH\nload_trainable(INITIAL);probe_opt.zero_grad(set_to_none=True);assert state_hash(trainable_state())==INITIAL_HASH\nwrite_json(probe,FT_ROOT/\'gradient_and_weight_update_preflight.json\')\ndel probe_opt,d,fx,S,feat,official,rendered,expected,raw,loss,vals,grads;gc.collect();torch.cuda.empty_cache()\nBOARD=[run_trial(spec) for spec in FT_TRIALS]\npd.DataFrame(BOARD).to_csv(FT_ROOT/\'leaderboard.csv\',index=False);display(pd.DataFrame(BOARD))\n# Lock immutable copies only after every trial has reached 4500 training updates.\nlocked={}\nfor row in BOARD:\n    t=row[\'trial\'];src=Path(row[\'checkpoint\']);dst=FT_ROOT/f\'locked_{t}.pt\'\n    if dst.exists():assert sha(dst)==sha(src),\'Locked checkpoint differs; do not overwrite after testing\'\n    else:shutil.copy2(src,dst)\n    locked[t]=dict(epoch=row[\'best_epoch\'],score=row[\'score\'],path=str(dst),sha256=sha(dst))\nselected=min(BOARD,key=lambda r:r[\'score\'])[\'trial\'] if min(r[\'score\'] for r in BOARD)<1 else \'Baseline\'\nlock=dict(protocol_id=FT_ID,selected_on_validation=selected,checkpoints=locked,test_pairs=BARBER_CASES[\'test_candidate\'])\npath=FT_ROOT/\'locked_before_test.json\'\nif path.exists():assert json.loads(path.read_text())==lock\nelse:write_json(lock,path)\nbs.cpu();gc.collect();torch.cuda.empty_cache()\nprint(\'Locked best checkpoints. Test does not select or modify shared weights.\')\n'
# Research adaptation: train shared synthesis weights, not per-image latents.
TRAIN_KEYS=list(REFERENCE_META['config']['train_keys'])
assert len(TRAIN_KEYS)==150 and len(set(TRAIN_KEYS))==150 and set(TRAIN_KEYS).isdisjoint(VAL_KEYS)
SEED=int(REFERENCE_META['config']['seed']);EPOCHS=30
for t,m in MANIFESTS.items():
    assert m['config']['train_keys']==TRAIN_KEYS and m['config']['validation_keys']==VAL_KEYS
    assert m['config']['epochs']==30 and m['config']['batch_size']==1
# LR values reproduce the numerical A-D grid, but parameter groups have different meanings.
FT_TRIALS=[dict(name=t,synthesis_lr=float(MANIFESTS[t]['config']['blend_lr']),rgb_lr=float(MANIFESTS[t]['config']['pp_lr'])) for t in 'ABCD']
FT_PROTOCOL=dict(method='Barbershop-FT: frozen latent front end + shared late synthesis fine-tuning',
    version=2,implementation_sha256=FT_IMPLEMENTATION_HASH,epochs=EPOCHS,batch_size=1,seed=SEED,trials=FT_TRIALS,
    train_keys=TRAIN_KEYS,validation_keys=VAL_KEYS,optimizer='Adam',betas=[.9,.999],gradient_clip=1.,
    trainable=['convs.6 onward (64-1024px)','to_rgbs.3 onward (64-1024px)'],
    frozen=['mapping','early synthesis','inversion','alignment','blending latent optimizer','ArcFace','parser'],
    noise='fixed pretrained noise buffers in front end and decoder',
    weights={k:float(WEIGHTS[k]) for k in ['final_color','final_shape','identity','protected','detail']},
    omitted_HFG_terms=['pre_color','pre_shape','latent_anchor: no corresponding trainable HFG stages'],
    detail_reference='frozen Barbershop own baseline; absolute MAE gate 0.02, declared before training',
    gates=dict(identity_drop=.02,protected_increase=.01,detail_absolute_increase=.02),
    selection='validation only; normalized color+shape, eligibility gates; baseline fallback',
    shape_reference='HFG-derived proxy, not independent annotated ground truth',
    source_manifest_sha256=sha(SOURCE/'manifest.json'),barbershop_revision=BARBER_REV,
    ffhq_sha256=sha(BARBER_WEIGHTS/'ffhq.pt'),seg_sha256=sha(BARBER_WEIGHTS/'seg.pth'),
    training_script_sha256=hashlib.sha256(FT_TRAIN_SOURCE.encode()).hexdigest())
for key in TRAIN_KEYS+VAL_KEYS:
    assert sha(SOURCE/'cache'/(key+'.pt'))==REFERENCE_META['source_cache_hashes'][key],key
FT_PROTOCOL['source_cache_hashes']={k:REFERENCE_META['source_cache_hashes'][k] for k in TRAIN_KEYS+VAL_KEYS}
# All masks must match the original A-D run, including train masks.
mask_hashes={}
for key in TRAIN_KEYS+VAL_KEYS:
    mask_hashes[key]={}
    for field in ['eval_core','eval_refmask','target','protected']:
        h=sha(STUDY_ROOT/'trial_A/inputs'/key/(field+'.png'))
        assert all(sha(STUDY_ROOT/f'trial_{t}/inputs'/key/(field+'.png'))==h for t in 'BCD')
        mask_hashes[key][field]=h
FT_PROTOCOL['mask_hashes']=mask_hashes
FT_ID=hashlib.sha256(json.dumps(FT_PROTOCOL,sort_keys=True).encode()).hexdigest()[:12]
FT_ROOT=Path('/content/drive/MyDrive/Barbershop_FT_30Epoch')/FT_ID
FT_ROOT.mkdir(parents=True,exist_ok=True)
def write_json(obj,path):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2));os.replace(tmp,path)
protocol_path=FT_ROOT/'protocol.json'
if protocol_path.exists():assert json.loads(protocol_path.read_text())==FT_PROTOCOL
else:write_json(FT_PROTOCOL,protocol_path)
BARBER_ROOT=FT_ROOT/'front_end';BARBER_ROOT.mkdir(exist_ok=True)
BARBER_TRIALS=[dict(name='BS_Pretrained',align_lr=.01,blend_lr=.01)]
by_current_key={r['key']:r for r in trainval_cases}
BARBER_CASES={split:[dict(key=k,**{role:resolve_dataset_id(by_current_key[k][role],role) for role in ['face','shape','color']}) for k in keys] for split,keys in [('train',TRAIN_KEYS),('validation',VAL_KEYS)]}
BARBER_CASES['test_candidate']=[{k:r[k] for k in ['key','face','shape','color']} for r in TEST_CASES]
assert all(by_current_key[k]['split']=='train' for k in TRAIN_KEYS)
assert all(by_current_key[k]['split']=='validation' for k in VAL_KEYS)
# Dataset audit already checks test IDs/pixel hashes/groups against all train+validation roles.
# Also verify train/val face-identity separation when the supplied dataset declares groups.
face_groups=lambda keys:{ITEM_BY_ID[resolve_dataset_id(by_current_key[k]['face'],'face')]['group_id'] for k in keys}
assert face_groups(TRAIN_KEYS).isdisjoint(face_groups(VAL_KEYS)), 'Train/val face identities overlap; resolve the split before claiming held-out validation'
CROPS=BARBER_ROOT/'shared_crops';CROPS.mkdir(exist_ok=True)

def hfg_offload():
    for module in [enc,pp,G,align,id_net,parser_net,hair_fast.embed.encoder,hair_fast.embed.e4e]:module.cpu()
    gc.collect();torch.cuda.empty_cache()

def prepare_frontend(split):
    # Test front end is prepared only after all four best checkpoints are locked.
    if split=='test_candidate':assert (FT_ROOT/'locked_before_test.json').is_file()
    root=BARBER_ROOT/split;root.mkdir(exist_ok=True)
    crop_checks=[];small={}
    for ident in sorted({r[role] for r in BARBER_CASES[split] for role in ['face','shape','color']}):
        path=CROPS/(ident+'.png')
        if not path.exists():
            with Image.open(DATA_ROOT/ITEM_BY_ID[ident]['file']) as im:original=im.convert('RGB').copy()
            tensor=align_face([original])[0].unsqueeze(0);assert tensor.shape==(1,3,1024,1024)
            saveim(tensor,path)
        with Image.open(path) as im:array=np.asarray(im.convert('RGB')).copy()
        tensor=torch.from_numpy(array).permute(2,0,1)[None].float()/255
        small[ident]=hair_fast.embed.downsample_256(tensor.cuda()).cpu()
    for row in BARBER_CASES[split]:
        d=load_case(row['key']) if split!='test_candidate' else torch.load(TEST_CACHE/(row['key']+'.pt'),map_location='cpu',weights_only=True)
        for role,field in [('face','face'),('color','ref')]+([('shape','shape')] if split=='test_candidate' else []):
            delta=float((small[row[role]]-d[field].cpu()).abs().max())
            assert delta<=1e-3,f'Crop mismatch {split}/{row["key"]}/{role}: {delta}'
            crop_checks.append(dict(case=row['key'],role=role,max_abs_delta=delta))
    pd.DataFrame(crop_checks).to_csv(root/'crop_checks.csv',index=False);del small,d
    cfg=dict(root=str(root),input_dir=str(CROPS),ckpt=str(BARBER_WEIGHTS/'ffhq.pt'),seg=str(BARBER_WEIGHTS/'seg.pth'),pca=str(BARBER_WEIGHTS/'ffhq_PCA.npz'),trials=BARBER_TRIALS,cases=BARBER_CASES[split],split=split,
             seed=SEED,protocol_id=FT_ID,crop_hashes={p.name:sha(p) for p in [CROPS/(i+'.png') for i in {r[role] for r in BARBER_CASES[split] for role in ['face','shape','color']}]})
    path=root/'worker_config.json'
    if path.exists():assert json.loads(path.read_text())==cfg
    else:write_json(cfg,path)
    hfg_offload()
    with (root/'worker.log').open('a') as log:
        proc=subprocess.Popen([sys.executable,str(BARBER_REPO/'experiment_worker.py'),str(path)],cwd=BARBER_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1'})
        for line in proc.stdout:log.write(line);log.flush();print(line,end='')
        if proc.wait():raise RuntimeError('Front end failed; completed cases are retained. Rerun this cell.')
    assert json.loads((root/'completion.json').read_text())['rows']==len(BARBER_CASES[split])
    id_net.cuda().eval();parser_net.cuda().eval()

display(pd.DataFrame(FT_TRIALS))
print('New shared-weight experiment:',FT_ROOT,'| 4 x 30 x 150 = 18,000 optimizer steps, plus latent preprocessing')
prepare_frontend('train')
prepare_frontend('validation')


## ฝึก shared weights จริง: A–D × 30 epochs
ตรวจ gradient และ weight delta ของทั้งสอง parameter groups ก่อนฝึก แล้ว reset กลับ pretrained เพื่อไม่ให้นับ preflight เป็นการฝึก trial

In [ ]:
exec(compile(FT_TRAIN_SOURCE,'barbershop_shared_weight_training','exec'))


## ล็อก best → test → กราฟและภาพเปรียบเทียบ
ห้ามปรับค่าฝึกหลังเห็น test ผลของทั้งสี่ trials แสดงเพื่อรายงาน ไม่ใช้เลือกผู้ชนะใหม่

In [ ]:
FT_REPORT=LOCAL_REPORT/'04_barbershop_FT';FT_REPORT.mkdir(exist_ok=True)
prepare_frontend('test_candidate')
bs.cuda();id_net.cuda().eval();parser_net.cuda().eval()
FT_METRICS=[]
for split,keys in [('validation',VAL_KEYS),('test_candidate',[r['key'] for r in BARBER_CASES['test_candidate']])]:
    for model in ['Baseline','A','B','C','D']:
        if model=='Baseline':load_trainable(INITIAL);epoch=0
        else:
            entry=lock['checkpoints'][model];assert sha(entry['path'])==entry['sha256']
            ck=torch.load(entry['path'],map_location='cpu',weights_only=True)
            assert ck['protocol_id']==FT_ID and ck['trial']['name']==model
            load_trainable(ck['state_dict']);epoch=ck['epoch'];del ck
        before=state_hash(trainable_state())
        FT_METRICS.extend(evaluate_bs(keys,split,epoch,model,save_images=True))
        assert before==state_hash(trainable_state()) and frozen_hash()==FROZEN_HASH
    frame=pd.DataFrame([r for r in FT_METRICS if r['split']==split])
    assert len(frame)==5*len(keys) and not frame.duplicated(['model','case']).any()
    frame.to_csv(FT_REPORT/split/'per_case_metrics.csv',index=False)
    display(frame.groupby('model').mean(numeric_only=True))
assert json.loads((FT_ROOT/'locked_before_test.json').read_text())==lock

from matplotlib.backends.backend_pdf import PdfPages
metrics=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),
    ('target_mask_IoU','HFG-derived target-mask agreement ↑'),('identity_cosine','Identity cosine ↑'),
    ('protected_RGB_MAE_vs_input','Protected RGB MAE ↓'),('protected_SSIM','Protected SSIM ↑')]
with PdfPages(FT_REPORT/'Barbershop_FT_vs_HairFastGAN.pdf') as pdf:
    fig,ax=plt.subplots(figsize=(11,8));ax.axis('off')
    notes='\n'.join([
        'Barbershop-FT: a research adaptation, NOT original Barbershop training',
        'Shared late synthesis weights trained: 150 pairs x 30 epochs x 4 trials.',
        'Frozen latent front end. Fixed pretrained noise. Baseline plus four validation-selected checkpoints.',
        'Same pairs, masks and common evaluation functions as the HairFastGAN experiment.',
        'Architectures, parameter counts, full losses and compute budgets are NOT identical.',
        'BS-FT omits HFG intermediate-stage/latent-anchor losses; preserves its own baseline detail.',
        'IoU measures agreement with HFG-derived target masks, NOT independent ground truth.',
        'Test remains candidate holdout: older development-data overlap is not fully verified.',
        'SD across cases is not a confidence interval. Do not select a winner from test.',
        'Train curves are minibatch observations; validation is after each epoch.',
        'A best epoch of 0 means no trained checkpoint passed the declared validation rules.',
        'Cross-model plots omit own-baseline detail differences because references differ.',
        'Timing: front-end optimization and shared-weight training reported separately.',
        f'Protocol: {FT_ID} | validation-selected BS condition: {selected}'
    ])
    ax.text(.02,.98,notes,va='top',fontsize=11,linespacing=1.8);pdf.savefig(fig);plt.close(fig)
    # First show each trial's own train/validation curves, then the combined four-trial curves.
    histories={}
    for t in 'ABCD':
        source=FT_ROOT/f'trial_{t}';dest=FT_REPORT/f'trial_{t}';dest.mkdir(exist_ok=True)
        for name in ['training_steps.csv','validation_per_case.csv','manifest.json']:shutil.copy2(source/name,dest/name)
        steps=pd.read_csv(source/'training_steps.csv');val=pd.read_csv(source/'validation_per_case.csv')
        assert len(steps)==4500 and steps['update'].nunique()==4500 and len(val)==279
        assert steps.groupby('epoch').size().eq(150).all() and val.groupby('epoch').size().eq(9).all()
        train=steps.groupby('epoch').mean(numeric_only=True);valid=val.groupby('epoch').mean(numeric_only=True)
        histories[t]=(train,valid)
        specs=[('loss','BS-FT objective'),('L_quantile','L error'),('ab_quantile','ab error'),
               ('identity_cosine','Identity cosine'),('protected_RGB_MAE_vs_input','Protected MAE'),
               ('detail_512_MAE_vs_own_baseline','Detail MAE vs own frozen baseline'),
               ('target_mask_IoU','Target-mask agreement'),('shape_loss','Differentiable shape loss')]
        fig,axes=plt.subplots(2,4,figsize=(19,8))
        for ax,(metric,label) in zip(axes.flat,specs):
            ax.plot(train.index,train[metric],label='Train: mean of updates')
            ax.plot(valid.index,valid[metric],label='Validation: end of epoch')
            ax.set_title(label);ax.set_xlabel('Epoch');ax.grid(alpha=.25);ax.legend(fontsize=8)
        fig.suptitle(f'Barbershop-FT trial {t} | separate train / validation')
        fig.tight_layout();fig.savefig(dest/'train_val.png',dpi=150);pdf.savefig(fig);plt.show();plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(13,4))
    for t,(tr,va) in histories.items():
        axes[0].plot(tr.index,tr.loss,label=t);axes[1].plot(va.index,va.loss,label=t)
    for ax,label in zip(axes,['Train objective: average of updates','Validation objective: end of epoch']):
        ax.set_title(label);ax.set_xlabel('Epoch');ax.grid(alpha=.25);ax.legend()
    fig.tight_layout();fig.savefig(FT_REPORT/'all_trials_train_validation.png',dpi=160);pdf.savefig(fig);plt.show();plt.close(fig)
    # Common metrics across ten models: five HFG and five BS-FT conditions.
    for split in ['validation','test_candidate']:
        bsframe=pd.DataFrame([r for r in FT_METRICS if r['split']==split]);bsframe['model']='BS_'+bsframe['model']
        hfroot=LOCAL_REPORT/('02_reevaluation' if split=='validation' else '03_test')
        hf=pd.read_csv(hfroot/'per_case_metrics.csv');hf['model']='HF_'+hf['model']
        columns=['case','model']+[k for k,_ in metrics]
        merged=pd.concat([hf[columns],bsframe[columns]],ignore_index=True)
        order=['HF_'+t for t in ['Baseline','A','B','C','D']]+['BS_'+t for t in ['Baseline','A','B','C','D']]
        expected={r['key'] for r in BARBER_CASES[split]}
        for model in order:assert set(merged[merged.model==model]['case'])==expected
        avg=merged.groupby('model').mean(numeric_only=True).reindex(order);sd=merged.groupby('model').std(numeric_only=True).reindex(order)
        merged.to_csv(FT_REPORT/f'{split}_common_metrics.csv',index=False)
        avg.to_csv(FT_REPORT/f'{split}_means.csv');sd.to_csv(FT_REPORT/f'{split}_SD.csv')
        fig,axes=plt.subplots(2,3,figsize=(17,10))
        for ax,(metric,label) in zip(axes.flat,metrics):
            ax.errorbar(range(10),avg[metric],yerr=sd[metric],fmt='o',capsize=3)
            ax.set_xticks(range(10),order,rotation=55,ha='right');ax.set_title(label);ax.grid(alpha=.25)
        fig.suptitle(f'{split}: same cases | mean ± SD | BS = adapted Barbershop-FT')
        fig.tight_layout();fig.savefig(FT_REPORT/f'{split}_all_models.png',dpi=160);pdf.savefig(fig);plt.show();plt.close(fig)
        for row in BARBER_CASES[split]:
            key=row['key']
            panels=[(CROPS/(row[r]+'.png'),name) for r,name in [('face','Input'),('shape','Shape ref'),('color','Color ref')]]
            panels += [(hfroot/'images'/key/(t+'.png'),'HF '+t) for t in ['Baseline','A','B','C','D']]
            panels += [(FT_REPORT/split/'images'/key/(t+'.png'),'BS-FT '+t) for t in ['Baseline','A','B','C','D']]
            fig,axes=plt.subplots(4,4,figsize=(14,14))
            for ax in axes.flat:ax.axis('off')
            for ax,(path,label) in zip(axes.flat,panels):
                with Image.open(path) as im:ax.imshow(im)
                ax.set_title(label)
            fig.suptitle(split+' | '+key);fig.tight_layout();fig.savefig(FT_REPORT/f'{split}_{key}.png',dpi=130);pdf.savefig(fig);plt.show();plt.close(fig)

for name in ['protocol.json','environment_and_parameters.json','gradient_and_weight_update_preflight.json','leaderboard.csv','locked_before_test.json']:
    shutil.copy2(FT_ROOT/name,FT_REPORT/name)
times=[]
for split in ['train','validation','test_candidate']:
    rows=json.loads((BARBER_ROOT/split/'results.json').read_text())
    times.append(dict(stage='front_end_'+split,seconds=sum(r['cached_pipeline_seconds'] for r in rows)))
    inv=list((BARBER_ROOT/split/'inversion').glob('*_complete.json'))
    times.append(dict(stage='inversion_'+split,seconds=sum(json.loads(p.read_text())['total_seconds'] for p in inv)))
for t in 'ABCD':times.append(dict(stage='shared_weight_updates_'+t,seconds=float(pd.read_csv(FT_ROOT/f'trial_{t}/training_steps.csv').seconds.sum())))
pd.DataFrame(times).to_csv(FT_REPORT/'separate_compute_times.csv',index=False)
write_json(dict(status='complete',protocol_id=FT_ID,training_updates_per_trial=4500,validation_selected=selected,
    checkpoint_directory=str(FT_ROOT),test_used_for_training=False,test_used_for_selection=False),FT_REPORT/'completion.json')
# Report files are small; checkpoints and expensive front-end artifacts already reside in Drive.
DEST=FT_ROOT/'report';shutil.copytree(FT_REPORT,DEST,dirs_exist_ok=True)
REPORT_ZIP=Path(shutil.make_archive(str(FT_ROOT/'Barbershop_FT_report'),'zip',root_dir=FT_REPORT))
print('Report:',REPORT_ZIP,'| best/last/final_epoch30 checkpoints:',FT_ROOT)
files.download(str(REPORT_ZIP))
